# MNIST Analysis

Dataset-focused notebook for MNIST/InfiMNIST analysis and plots.




## Environment Setup


In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path('/home/abhuiyan/nnet_error_project').resolve()
DATASET_ROOT = REPO_ROOT / 'noisy_mnist'
os.chdir(DATASET_ROOT)
print('Working directory:', Path.cwd())



In [ ]:
from pathlib import Path

print('MNIST data dirs:')
for p in sorted((Path('data')).glob('*')):
    print(' -', p)

print('
Sample result files:')
for p in sorted(Path('results/data').glob('*.csv'))[:20]:
    print(' -', p)



# MNIST Noise Sensitivity
Load aggregated results and visualize test accuracy vs sigma for each activation.


## Load and Summarize Baseline MNIST Results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
summary_path = 'results/data/results_summary.csv'
df = pd.read_csv(summary_path)
x_col = 'p'
if 'corruption_mode' in df.columns and df['corruption_mode'].nunique() == 1:
    if df['corruption_mode'].iloc[0] == 'additive':
        x_col = 'sigma'
df


In [ ]:
model_types = sorted(df['model_type'].unique())
for model_type in model_types:
    sub = df[df['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    n = len(activations)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d = sub[sub['activation'] == act].sort_values(x_col)
        ax.errorbar(d[x_col], d['mean_test_accuracy'], yerr=d['stderr_test_accuracy'], marker='o')
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel(x_col)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    plt.tight_layout()
    plt.show()


## Explicit 3-subplot snippet (one per activation) for a chosen model type


In [ ]:
# Explicit 3-subplot snippet (one per activation) for a chosen model type.
model_type = df['model_type'].unique()[0]
sub = df[df['model_type'] == model_type]
activations = sorted(sub['activation'].unique())[:3]

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, act in zip(axes, activations):
    d = sub[sub['activation'] == act].sort_values(x_col)
    ax.errorbar(d[x_col], d['mean_test_accuracy'], yerr=d['stderr_test_accuracy'], marker='o')
    ax.set_title(f'{model_type} / {act}')
    ax.set_xlabel(x_col)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('mean test accuracy')
plt.tight_layout()
plt.show()


## Std-dev vs p for each activation (one subplot per activation)


In [ ]:
# Std-dev vs p for each activation (one subplot per activation).
model_type = df['model_type'].unique()[0]
sub = df[df['model_type'] == model_type]
activations = sorted(sub['activation'].unique())
n = len(activations)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
if n == 1:
    axes = [axes]
for ax, act in zip(axes, activations):
    d = sub[sub['activation'] == act].sort_values(x_col)
    ax.plot(d[x_col], d['std_test_accuracy'], marker='o')
    ax.set_title(f'{model_type} / {act}')
    ax.set_xlabel(x_col)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('std test accuracy')
plt.tight_layout()
plt.show()


## Corruption Visualization and Animation Setup


In [ ]:
import torch
from torchvision import datasets, transforms

mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
means = []
maxes = []
medians = []
modes = []
for x, _ in mnist:
    flat = x.flatten()
    means.append(flat.mean().item())
    maxes.append(flat.max().item())
    nonzero = flat[flat > 0]
    if nonzero.numel() == 0:
        medians.append(0.0)
        modes.append(0.0)
    else:
        medians.append(nonzero.median().item())
        modes.append(torch.mode(nonzero).values.item())

plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1)
plt.hist(means, bins=50, color='steelblue', alpha=0.8)
plt.title('MNIST image mean (per image)')
plt.xlabel('mean pixel value')
plt.ylabel('count')

plt.subplot(2, 2, 2)
plt.hist(maxes, bins=50, color='salmon', alpha=0.8)
plt.title('MNIST image max (per image)')
plt.xlabel('max pixel value')
plt.ylabel('count')

plt.subplot(2, 2, 3)
plt.hist(medians, bins=50, color='seagreen', alpha=0.8)
plt.title('MNIST image median (per image)')
plt.xlabel('median pixel value')
plt.ylabel('count')

plt.subplot(2, 2, 4)
plt.hist(modes, bins=50, color='slateblue', alpha=0.8)
plt.title('MNIST image mode (per image)')
plt.xlabel('mode pixel value')
plt.ylabel('count')
plt.tight_layout()
plt.show()


In [ ]:
import random

mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
idx = random.randrange(len(mnist))
img, label = mnist[idx]
plt.figure(figsize=(3, 3))
plt.imshow(img.squeeze(0), cmap='gray')
plt.title(f'Random MNIST sample (label={label})')
plt.axis('off')
plt.show()


In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Pick a single MNIST "4" and animate corruption with p in [0, 1].
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
idx = next(i for i, (_, y) in enumerate(mnist) if y == 8)
img, label = mnist[idx]

# New global seed per trajectory.
seed = random.randrange(1, 1_000_000_000)
random.seed(seed)
torch.manual_seed(seed)
print(f'Animation seed: {seed}')

ps = torch.linspace(0.0, 1.0, 100)
frames = []
for p in ps:
    mask = torch.rand_like(img) < p
    replacement = torch.rand_like(img)
    corrupted = torch.where(mask, replacement, img)
    frames.append(corrupted.squeeze(0).numpy())

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(frames[0], cmap='gray', vmin=0, vmax=1)
title = ax.set_title(f'label={label}, p={ps[0].item():.2f}')
ax.axis('off')

def update(frame_idx: int):
    im.set_data(frames[frame_idx])
    title.set_text(f'label={label}, p={ps[frame_idx].item():.2f}')
    return im, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=50,
    blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Average over multiple trajectories (each with its own global seed).
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
idx = next(i for i, (_, y) in enumerate(mnist) if y == 4)
img, label = mnist[idx]

num_trajectories = 1000
ps = torch.linspace(0.0, 1.0, 100)
sum_frames = torch.zeros((len(ps), 1, 28, 28))
seeds = [random.randrange(1, 1_000_000_000) for _ in range(num_trajectories)]

with torch.no_grad():
    for seed in seeds:
        torch.manual_seed(seed)
        for i, p in enumerate(ps):
            mask = torch.rand_like(img) < p
            replacement = torch.rand_like(img)
            corrupted = torch.where(mask, replacement, img)
            sum_frames[i] += corrupted

avg_frames = (sum_frames / num_trajectories).squeeze(1).numpy()

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(avg_frames[0], cmap='gray', vmin=0, vmax=1)
title = ax.set_title(f'label={label}, p={ps[0].item():.2f} (avg)')
ax.axis('off')

def update(frame_idx: int):
    im.set_data(avg_frames[frame_idx])
    title.set_text(f'label={label}, p={ps[frame_idx].item():.2f} (avg)')
    return im, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(avg_frames),
    interval=50,
    blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Additive noise animation: iid Uniform(-sigma/2, sigma/2) with clipping.
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
idx = next(i for i, (_, y) in enumerate(mnist) if y == 8)
img, label = mnist[idx]

sigmas = torch.linspace(0.0, 1.0, 100)
frames = []
for sigma in sigmas:
    noise = (torch.rand_like(img) - 0.5) * sigma
    corrupted = torch.clamp(img + noise, 0.0, 1.0)
    frames.append(corrupted.squeeze(0).numpy())

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(frames[0], cmap='gray', vmin=0, vmax=1)
title = ax.set_title(f'label={label}, sigma={sigmas[0].item():.2f}')
ax.axis('off')

def update(frame_idx: int):
    im.set_data(frames[frame_idx])
    title.set_text(f'label={label}, sigma={sigmas[frame_idx].item():.2f}')
    return im, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=50,
    blit=False,
)
plt.close(fig)
anim.save('mnist_additive_noise.gif', writer='pillow', fps=20)
HTML(anim.to_jshtml())


## Replacement corruption quiz: set p, guess the digit, then log results


In [ ]:
# Replacement corruption quiz: set p, guess the digit, then log results.
import random
import os
import pandas as pd

p = 0.9  # set corruption probability here
num_trials = 3  # number of quiz rounds to run
log_path = 'results/data/guess_log.csv'
os.makedirs('results', exist_ok=True)

if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    trial = int(df['trial'].max()) + 1 if not df.empty else 1
else:
    df = pd.DataFrame(columns=['trial', 'p', 'guess', 'true', 'success'])
    trial = 1

mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())

for _ in range(num_trials):
    idx = random.randrange(len(mnist))
    img, label = mnist[idx]
    mask = torch.rand_like(img) < p
    replacement = torch.rand_like(img)
    corrupted = torch.where(mask, replacement, img)

    plt.figure(figsize=(3, 3))
    plt.imshow(corrupted.squeeze(0), cmap='gray', vmin=0, vmax=1)
    plt.title(f'Replacement corruption (p={p:.2f})')
    plt.axis('off')
    plt.show()

    guess = input('Your guess (0-9): ')
    try:
        guess_val = int(guess)
    except ValueError:
        guess_val = -1

    success = 1 if guess_val == int(label) else 0
    new_row = {
        'trial': trial,
        'p': p,
        'guess': guess_val,
        'true': int(label),
        'success': success,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(log_path, index=False)

    print(f'Label: {label}')
    trial += 1

df.tail()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Finite-size scaling from per-run width sweep data.
per_run_path = 'results/data/results_per_run_width_sweep_21_steps_high_p.csv'
df = pd.read_csv(per_run_path)

# Collapse MLP data only (width encoded in mlp_hidden_sizes).
df = df[df['model_type'] == 'mlp'].copy()
df['mlp_width'] = df['mlp_hidden_sizes'].apply(lambda x: int(str(x).strip('[]').split(',')[0]))

# Aggregate mean accuracy by width, activation, and p.
grouped = df.groupby(['activation', 'mlp_width', 'p'], as_index=False)['test_accuracy']
agg = grouped.mean().rename(columns={'test_accuracy': 'mean_test_accuracy'})

# Define p* as the p where accuracy falls to half of (A0 - Achance).
chance = 0.1
pstars = {}
widths = sorted(agg['mlp_width'].unique())
activations = sorted(agg['activation'].unique())

for act in activations:
    pstars[act] = {}
    for w in widths:
        sub = agg[(agg['activation'] == act) & (agg['mlp_width'] == w)].sort_values('p')
        a0 = sub[sub['p'] == sub['p'].min()]['mean_test_accuracy'].iloc[0]
        target = chance + 0.5 * (a0 - chance)
        idx = (sub['mean_test_accuracy'] - target).abs().idxmin()
        pstars[act][w] = sub.loc[idx, 'p']

# Choose a scaling exponent alpha by a simple heuristic.
alpha = 0.5

for act in activations:
    fig, ax = plt.subplots(figsize=(4, 3))
    for w in widths:
        sub = agg[(agg['activation'] == act) & (agg['mlp_width'] == w)].sort_values('p')
        p_star = pstars[act][w]
        x = (sub['p'] - p_star) * (w ** alpha)
        y = sub['mean_test_accuracy']
        ax.plot(x, y, marker='o', label=f'w={w}')
    ax.set_title(f'Collapse: {act} (alpha={alpha})')
    ax.set_xlabel('(p - p*) * w^alpha')
    ax.set_ylabel('mean test accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# Fit sigmoid to accuracy vs p and estimate p* and slope.
per_run_path = 'results/data/results_per_run_width_sweep_21_steps_high_p.csv'
df = pd.read_csv(per_run_path)
df = df[df['model_type'] == 'mlp'].copy()
df['mlp_width'] = df['mlp_hidden_sizes'].apply(lambda x: int(str(x).strip('[]').split(',')[0]))

grouped = df.groupby(['activation', 'mlp_width', 'p'], as_index=False)['test_accuracy']
agg = grouped.mean().rename(columns={'test_accuracy': 'mean_test_accuracy'})

chance = 0.1
activations = sorted(agg['activation'].unique())
widths = sorted(agg['mlp_width'].unique())

fit_results = {}
for act in activations:
    fit_results[act] = {}
    for w in widths:
        sub = agg[(agg['activation'] == act) & (agg['mlp_width'] == w)].sort_values('p')
        p_vals = sub['p'].values.reshape(-1, 1)
        a_vals = sub['mean_test_accuracy'].values
        # Normalize accuracy into (0,1) for logistic fit.
        a0 = a_vals[0]
        y = (a_vals - chance) / max(1e-6, (a0 - chance))
        y = np.clip(y, 1e-4, 1 - 1e-4)
        logit_y = np.log(y / (1 - y))

        # Fit logit_y ~ beta0 + beta1 * p
        X = np.hstack([np.ones_like(p_vals), p_vals])
        beta, _, _, _ = np.linalg.lstsq(X, logit_y, rcond=None)
        beta0, beta1 = beta
        p_star = -beta0 / beta1
        slope = abs(beta1)

        fit_results[act][w] = {
            'p_star': float(p_star),
            'slope': float(slope),
            'a0': float(a0),
        }

# Plot p* and slope vs width.
for act in activations:
    pstars = [fit_results[act][w]['p_star'] for w in widths]
    slopes = [fit_results[act][w]['slope'] for w in widths]

    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].plot(widths, pstars, marker='o')
    ax[0].set_title(f'p* vs width ({act})')
    ax[0].set_xlabel('width')
    ax[0].set_ylabel('p*')
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(widths, slopes, marker='o')
    ax[1].set_title(f'slope vs width ({act})')
    ax[1].set_xlabel('width')
    ax[1].set_ylabel('sigmoid slope')
    ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Alpha scan for best collapse (minimize variance near knee).
alphas = np.linspace(0.0, 1.0, 21)
alpha_best = {}

for act in activations:
    scores = []
    for alpha in alphas:
        xs = []
        ys = []
        for w in widths:
            sub = agg[(agg['activation'] == act) & (agg['mlp_width'] == w)].sort_values('p')
            p_star = fit_results[act][w]['p_star']
            x = (sub['p'].values - p_star) * (w ** alpha)
            y = sub['mean_test_accuracy'].values
            xs.append(x)
            ys.append(y)
        # Align by interpolating onto a common grid.
        x_min = max([x.min() for x in xs])
        x_max = min([x.max() for x in xs])
        grid = np.linspace(x_min, x_max, 50)
        interp = [np.interp(grid, xs[i], ys[i]) for i in range(len(xs))]
        stack = np.vstack(interp)
        scores.append(np.mean(np.var(stack, axis=0)))
    alpha_best[act] = alphas[int(np.argmin(scores))]

# Plot best collapse per activation.
for act in activations:
    alpha = alpha_best[act]
    fig, ax = plt.subplots(figsize=(4, 3))
    for w in widths:
        sub = agg[(agg['activation'] == act) & (agg['mlp_width'] == w)].sort_values('p')
        p_star = fit_results[act][w]['p_star']
        x = (sub['p'].values - p_star) * (w ** alpha)
        y = sub['mean_test_accuracy'].values
        ax.plot(x, y, marker='o', label=f'w={w}')
    ax.set_title(f'Best collapse: {act} (alpha={alpha:.2f})')
    ax.set_xlabel('(p - p*) * w^alpha')
    ax.set_ylabel('mean test accuracy')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()



## Binary MNIST replacement corruption demo


In [ ]:
# Binary MNIST replacement corruption demo
import random
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

label_target = 8
p = 0.3  # corruption strength

transform = transforms.ToTensor()
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transform)
idx = next(i for i, (_, y) in enumerate(mnist) if y == label_target)
img, label = mnist[idx]

# Binarize image: >=0.5 -> 1, <0.5 -> 0
img_bin = (img >= 0.5).float()

# Replacement corruption: with prob p replace with Bernoulli(0.5)
mask = torch.rand_like(img_bin) < p
replacement = (torch.rand_like(img_bin) < 0.5).float()
corrupted = torch.where(mask, replacement, img_bin)

plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(img_bin.squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
plt.title(f'Binarized label={label}')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(corrupted.squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
plt.title(f'Corrupted p={p}')
plt.axis('off')

plt.tight_layout()
plt.show()



## Interactive binary replacement corruption (slider)


In [ ]:
# Interactive binary replacement corruption (slider)
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from ipywidgets import interact, FloatSlider, IntSlider

label_target = 8
transform = transforms.ToTensor()
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transform)
idx = next(i for i, (_, y) in enumerate(mnist) if y == label_target)
img, label = mnist[idx]
img_bin = (img >= 0.5).float()

def show_binary_corruption(p=0.3, seed=0):
    torch.manual_seed(seed)
    mask = torch.rand_like(img_bin) < p
    replacement = (torch.rand_like(img_bin) < 0.5).float()
    corrupted = torch.where(mask, replacement, img_bin)

    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img_bin.squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
    plt.title(f'Binarized label={label}')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(corrupted.squeeze(0).numpy(), cmap='gray', vmin=0, vmax=1)
    plt.title(f'Corrupted p={p:.2f}, seed={seed}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    show_binary_corruption,
    p=FloatSlider(min=0.0, max=1.0, step=0.01, value=0.3),
    seed=IntSlider(min=0, max=1000, step=1, value=0),
)



In [ ]:
from matplotlib import animation
from IPython.display import HTML
import random
import torch
from torchvision import datasets, transforms

# Binary MNIST replacement corruption: average over trajectories.
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
label_target = 8
idx = next(i for i, (_, y) in enumerate(mnist) if y == label_target)
img, label = mnist[idx]
img_bin = (img >= 0.5).float()

num_trajectories = 200
ps = torch.linspace(0.0, 1.0, 100)
sum_frames = torch.zeros((len(ps), 1, 28, 28))
seeds = [random.randrange(1, 1_000_000_000) for _ in range(num_trajectories)]

with torch.no_grad():
    for seed in seeds:
        torch.manual_seed(seed)
        for i, p in enumerate(ps):
            mask = torch.rand_like(img_bin) < p
            replacement = (torch.rand_like(img_bin) < 0.5).float()
            corrupted = torch.where(mask, replacement, img_bin)
            sum_frames[i] += corrupted

avg_frames = (sum_frames / num_trajectories).squeeze(1).numpy()

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(avg_frames[0], cmap='gray', vmin=0, vmax=1)
title = ax.set_title(f'label={label}, p={ps[0].item():.2f} (avg, binary)')
ax.axis('off')

def update(frame_idx: int):
    im.set_data(avg_frames[frame_idx])
    title.set_text(f'label={label}, p={ps[frame_idx].item():.2f} (avg, binary)')
    return im, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(avg_frames),
    interval=50,
    blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())



In [ ]:
from matplotlib import animation
from IPython.display import HTML
import random
import torch
from torchvision import datasets, transforms

# Binary MNIST replacement corruption: single trajectory.
mnist = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
label_target = 8
idx = next(i for i, (_, y) in enumerate(mnist) if y == label_target)
img, label = mnist[idx]
img_bin = (img >= 0.5).float()

seed = random.randrange(1, 1_000_000_000)
torch.manual_seed(seed)
print(f'Animation seed: {seed}')

ps = torch.linspace(0.0, 1.0, 100)
frames = []
for p in ps:
    mask = torch.rand_like(img_bin) < p
    replacement = (torch.rand_like(img_bin) < 0.5).float()
    corrupted = torch.where(mask, replacement, img_bin)
    frames.append(corrupted.squeeze(0).numpy())

fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(frames[0], cmap='gray', vmin=0, vmax=1)
title = ax.set_title(f'label={label}, p={ps[0].item():.2f} (binary)')
ax.axis('off')

def update(frame_idx: int):
    im.set_data(frames[frame_idx])
    title.set_text(f'label={label}, p={ps[frame_idx].item():.2f} (binary)')
    return im, title

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=50,
    blit=False,
)
plt.close(fig)
HTML(anim.to_jshtml())



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path('results/data')
# Pick the most recent pristine-fraction per-run CSV.
per_run_files = sorted(results_dir.glob('results_per_run_*pristine_fraction*.csv'))
if not per_run_files:
    raise FileNotFoundError('No results_per_run_*pristine_fraction*.csv files found in results/.')
per_run_path = max(per_run_files, key=lambda p: p.stat().st_mtime)
print(f'Loading: {per_run_path}')

rows = pd.read_csv(per_run_path)
summary = (
    rows.groupby(['activation', 'model_type', 'pristine_frac', 'p'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_accuracy=('train_accuracy', 'mean'),
        std_train_accuracy=('train_accuracy', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)

model_types = sorted(summary['model_type'].unique())
for model_type in model_types:
    sub = summary[summary['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    pristine_fracs = sorted(sub['pristine_frac'].unique())
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        act_sub = sub[sub['activation'] == act]
        for frac in pristine_fracs:
            d = act_sub[act_sub['pristine_frac'] == frac].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'pristine={frac:.2f}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path('results/data')
per_run_files = sorted(results_dir.glob('results_per_run_*pristine_fraction*.csv'))
if not per_run_files:
    raise FileNotFoundError('No results_per_run_*pristine_fraction*.csv files found in results/.')

# Prefer the newest file that already contains a 'p' column.
with_p = []
for f in per_run_files:
    try:
        cols = pd.read_csv(f, nrows=1).columns
        if 'p' in cols:
            with_p.append(f)
    except Exception:
        pass

if with_p:
    per_run_path = max(with_p, key=lambda p: p.stat().st_mtime)
else:
    per_run_path = max(per_run_files, key=lambda p: p.stat().st_mtime)

print(f'Loading: {per_run_path}')
rows = pd.read_csv(per_run_path)

if 'p' not in rows.columns:
    raise ValueError(
        "This results file has no 'p' column (it was created before the script fix). "
        "Please re-run run_experiments_noisy_training_data_pristine_fraction.py to regenerate results."
    )

summary = (
    rows.groupby(['activation', 'model_type', 'pristine_frac', 'p'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_accuracy=('train_accuracy', 'mean'),
        std_train_accuracy=('train_accuracy', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)

model_types = sorted(summary['model_type'].unique())
for model_type in model_types:
    sub = summary[summary['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    pristine_fracs = sorted(sub['pristine_frac'].unique())
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        act_sub = sub[sub['activation'] == act]
        for frac in pristine_fracs:
            d = act_sub[act_sub['pristine_frac'] == frac].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'pristine={frac:.2f}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_r20_e20_tf0.500_p0.75-1.00_w128-1024_d1_loss-cross_entropy_width_sweep.csv'
rows = pd.read_csv(per_run_path)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

model_types = sorted(summary['model_type'].unique())
for model_type in model_types:
    sub = summary[summary['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    widths = sorted(sub['mlp_width'].unique())
    n = len(activations)

    # Mean test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Std test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['std_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('std test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean test loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_loss'],
                yerr=d['stderr_test_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean train loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['mean_train_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean train loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_r20_e20_tf0.500_p0.75-1.00_w128-1024_d1_loss-cross_entropy_width_sweep.csv'
rows = pd.read_csv(per_run_path)

# Recover width from mlp_hidden_sizes (stored as string like "[128, 128]" or "[128]").
def parse_width(value: str) -> int:
    try:
        sizes = ast.literal_eval(value)
        if isinstance(sizes, list) and sizes:
            return int(sizes[0])
    except Exception:
        pass
    return None

rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

model_types = sorted(summary['model_type'].unique())
for model_type in model_types:
    sub = summary[summary['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    widths = sorted(sub['mlp_width'].dropna().unique())
    n = len(activations)

    # Mean test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Std test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['std_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('std test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean test loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_loss'],
                yerr=d['stderr_test_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean train loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['mean_train_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean train loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

per_run_path = Path('results/data/results_per_run_mlpws_r20_e20_tf0.500_p0.75-1.00_w128-1024_d1_loss-cross_entropy_width_sweep.csv')
rows = pd.read_csv(per_run_path)

# Recover width from mlp_hidden_sizes (stored as string like "[128, 128]" or "[128]").
def parse_width(value: str) -> int:
    try:
        sizes = ast.literal_eval(value)
        if isinstance(sizes, list) and sizes:
            return int(sizes[0])
    except Exception:
        pass
    return None

rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

cache_key = per_run_path.name.replace('results_per_run_', '').replace('.csv', '')
out_pdf = per_run_path.with_name(f'plots_{cache_key}.pdf')

with PdfPages(out_pdf) as pdf:
    model_types = sorted(summary['model_type'].unique())
    for model_type in model_types:
        sub = summary[summary['model_type'] == model_type]
        activations = sorted(sub['activation'].unique())
        widths = sorted(sub['mlp_width'].dropna().unique())
        n = len(activations)

        # Mean test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(
                    d['p'],
                    d['mean_test_accuracy'],
                    yerr=d['stderr_test_accuracy'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Std test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(
                    d['p'],
                    d['std_test_accuracy'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('std test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Mean test loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(
                    d['p'],
                    d['mean_test_loss'],
                    yerr=d['stderr_test_loss'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Mean train loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(
                    d['p'],
                    d['mean_train_loss'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean train loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f'Saved: {out_pdf}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary_path = 'results/data/results_summary_mlp_r50_e10_pristine0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_pristine_fraction.csv'
df = pd.read_csv(summary_path)

model_types = sorted(df['model_type'].unique())
for model_type in model_types:
    sub = df[df['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    n = len(activations)

    # Mean test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d = sub[sub['activation'] == act].sort_values('p')
        ax.errorbar(
            d['p'],
            d['mean_test_accuracy'],
            yerr=d['stderr_test_accuracy'],
            marker='o',
        )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    plt.tight_layout()
    plt.show()

    # Mean train accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d = sub[sub['activation'] == act].sort_values('p')
        ax.errorbar(
            d['p'],
            d['mean_train_accuracy'],
            yerr=d['stderr_train_accuracy'],
            marker='o',
        )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean train accuracy')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

summary_path = 'results/data/results_summary_mlp_r50_e10_pristine0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_pristine_fraction.csv'
df = pd.read_csv(summary_path)

model_types = sorted(df['model_type'].unique())
for model_type in model_types:
    sub = df[df['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    pristine_fracs = sorted(sub['pristine_frac'].unique())
    n = len(activations)

    # Mean test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        act_sub = sub[sub['activation'] == act]
        for frac in pristine_fracs:
            d = act_sub[act_sub['pristine_frac'] == frac].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'pristine={frac:.3f}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean train accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        act_sub = sub[sub['activation'] == act]
        for frac in pristine_fracs:
            d = act_sub[act_sub['pristine_frac'] == frac].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_train_accuracy'],
                yerr=d['stderr_train_accuracy'],
                marker='o',
                label=f'pristine={frac:.3f}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean train accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd

summary_path = "results/data/results_summary_mlp_r50_e10_pristine0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_pristine_fraction.csv"
df = pd.read_csv(summary_path)

acc_df = df[[
    "activation",
    "model_type",
    "p",
    "pristine_frac",
    "mean_test_accuracy",
    "mean_train_accuracy",
]].copy()

acc_df["abs_train_test_gap"] = (acc_df["mean_train_accuracy"] - acc_df["mean_test_accuracy"])

acc_df



## Finite-size scaling (FSS) on relu mean test accuracy for width sweep (cross-entropy)


In [ ]:
# Finite-size scaling (FSS) on relu mean test accuracy for width sweep (cross-entropy)
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = "results/data/results_per_run_mlpws_r20_e20_tf0.500_p0.75-1.00_w128-1024_d1_loss-cross_entropy_width_sweep.csv"
raw = pd.read_csv(per_run_path)
raw = raw[(raw["activation"] == "relu") & (raw["model_type"] == "mlp")].copy()

# Parse width from the serialized list (e.g. "[128]")
raw["mlp_width"] = raw["mlp_hidden_sizes"].apply(
    lambda s: int(ast.literal_eval(s)[0]) if isinstance(s, str) else int(s[0])
)

summary = (
    raw.groupby(["p", "mlp_width"], as_index=False)["test_accuracy"]
    .mean()
    .rename(columns={"test_accuracy": "mean_test_accuracy"})
)


def compute_fss_objective(data_by_width: dict[int, tuple[np.ndarray, np.ndarray]], pc: float, nu: float) -> float:
    curves = {}
    xs_all = []
    for width, (p_vals, s_vals) in data_by_width.items():
        if pc < p_vals.min() or pc > p_vals.max():
            continue
        s_pc = np.interp(pc, p_vals, s_vals)
        x_vals = (p_vals - pc) * (width ** (1.0 / nu))
        y_vals = s_vals - s_pc
        curves[width] = (x_vals, y_vals)
        xs_all.append(x_vals)
    if not curves:
        return float("inf")
    xs_all = np.unique(np.concatenate(xs_all))
    total = 0.0
    for x in xs_all:
        ys = []
        for x_vals, y_vals in curves.values():
            if x < x_vals.min() or x > x_vals.max():
                continue
            ys.append(np.interp(x, x_vals, y_vals))
        if len(ys) < 2:
            continue
        ybar = float(np.mean(ys))
        total += float(np.sum((np.array(ys) - ybar) ** 2))
    return total


def find_best_fss(act_df: pd.DataFrame) -> tuple[float, float]:
    widths = sorted(act_df["mlp_width"].unique())
    data_by_width = {}
    for width in widths:
        sub = act_df[act_df["mlp_width"] == width].sort_values("p")
        p_vals = sub["p"].to_numpy(dtype=float)
        s_vals = sub["mean_test_accuracy"].to_numpy(dtype=float)
        if len(p_vals) < 3:
            continue
        data_by_width[width] = (p_vals, s_vals)
    if len(data_by_width) < 2:
        return float("nan"), float("nan")

    p_min = min(v[0].min() for v in data_by_width.values())
    p_max = max(v[0].max() for v in data_by_width.values())
    pc_grid = np.linspace(p_min, p_max, 41)
    nu_grid = np.linspace(0.2, 5.0, 41)

    best = (float("inf"), np.nan, np.nan)
    for pc in pc_grid:
        for nu in nu_grid:
            r = compute_fss_objective(data_by_width, pc, nu)
            if r < best[0]:
                best = (r, pc, nu)

    _, pc0, nu0 = best
    if np.isnan(pc0) or np.isnan(nu0):
        return float("nan"), float("nan")

    pc_grid = np.linspace(max(p_min, pc0 - 0.05), min(p_max, pc0 + 0.05), 41)
    nu_grid = np.linspace(max(0.2, nu0 - 1.0), min(5.0, nu0 + 1.0), 41)
    best = (float("inf"), pc0, nu0)
    for pc in pc_grid:
        for nu in nu_grid:
            r = compute_fss_objective(data_by_width, pc, nu)
            if r < best[0]:
                best = (r, pc, nu)
    _, pc_opt, nu_opt = best
    return float(pc_opt), float(nu_opt)


pc_opt, nu_opt = find_best_fss(summary)
print(f"Best FSS fit: pc={pc_opt:.3f}, nu={nu_opt:.2f}")

# Plot raw curves
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for width in sorted(summary["mlp_width"].unique()):
    sub = summary[summary["mlp_width"] == width].sort_values("p")
    ax[0].plot(sub["p"], sub["mean_test_accuracy"], marker="o", label=f"w={width}")
ax[0].set_title("relu mean test accuracy vs p")
ax[0].set_xlabel("p")
ax[0].set_ylabel("mean test accuracy")
ax[0].grid(True, alpha=0.3)
ax[0].legend(fontsize=8)

# Plot FSS collapse
for width in sorted(summary["mlp_width"].unique()):
    sub = summary[summary["mlp_width"] == width].sort_values("p")
    p_vals = sub["p"].to_numpy(dtype=float)
    s_vals = sub["mean_test_accuracy"].to_numpy(dtype=float)
    if np.isnan(pc_opt) or np.isnan(nu_opt):
        continue
    s_pc = np.interp(pc_opt, p_vals, s_vals)
    x_vals = (p_vals - pc_opt) * (width ** (1.0 / nu_opt))
    y_vals = s_vals - s_pc
    ax[1].plot(x_vals, y_vals, marker="o", label=f"w={width}")
ax[1].set_title(f"FSS collapse (pc={pc_opt:.3f}, nu={nu_opt:.2f})")
ax[1].set_xlabel("(p - pc) * w^(1/nu)")
ax[1].set_ylabel("S(p,w) - S(pc,w)")
ax[1].grid(True, alpha=0.3)
ax[1].legend(fontsize=8)

plt.tight_layout()



## pyfssa (fssa) finite-size scaling + collapse quality on relu accuracy


In [ ]:
# pyfssa (fssa) finite-size scaling + collapse quality on relu accuracy
import ast
import numpy as np

# Compat patch for deprecated numpy aliases used by fssa
if not hasattr(np, "int"):
    np.int = int

import pandas as pd

# Monkeypatch for scipy>=1.11: fssa expects scipy.optimize.optimize internals
import importlib
try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    # Backfill removed symbols if needed
    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    # If wrap_function still missing, define a compatible shim
    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function

    # Diagnostics
    print("scipy.optimize.optimize has wrap_function:", hasattr(_opt_mod, "wrap_function"))
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

try:
    import fssa
except Exception as exc:
    raise ImportError(
        "fssa (pyfssa) not available in this kernel. Install with: pip install fssa"
    ) from exc

per_run_path = "results/data/results_per_run_mlpws_r20_e20_tf0.500_p0.75-1.00_w128-1024_d1_loss-cross_entropy_width_sweep.csv"
raw = pd.read_csv(per_run_path)
raw = raw[(raw["activation"] == "relu") & (raw["model_type"] == "mlp")].copy()

# Parse width from serialized list, e.g. "[128]"
raw["mlp_width"] = raw["mlp_hidden_sizes"].apply(
    lambda s: int(ast.literal_eval(s)[0]) if isinstance(s, str) else int(s[0])
)

summary = (
    raw.groupby(["mlp_width", "p"], as_index=False)["test_accuracy"]
    .agg(["mean", "std", "count"])    .reset_index()
    .rename(columns={"mean": "mean_acc", "std": "std_acc", "count": "n"})
)
summary["stderr"] = summary["std_acc"] / np.sqrt(summary["n"])

# Ensure strictly positive errors for fssa.quality
# Optional: restrict to a p-window around the knee to improve collapse
p_min, p_max = 0.85, 0.98
if p_min is not None and p_max is not None:
    summary = summary[(summary["p"] >= p_min) & (summary["p"] <= p_max)].copy()

min_nonzero = summary.loc[summary["stderr"] > 0, "stderr"].min()
summary["stderr"] = summary["stderr"].fillna(min_nonzero if np.isfinite(min_nonzero) else 1e-6)
summary["stderr"] = summary["stderr"].clip(lower=1e-6)

# Use only p-values present for all widths
widths = sorted(summary["mlp_width"].unique())
common_p = None
for w in widths:
    pset = set(summary.loc[summary["mlp_width"] == w, "p"].values)
    common_p = pset if common_p is None else (common_p & pset)
ps = sorted(common_p) if common_p else sorted(summary["p"].unique())
summary = summary[summary["p"].isin(ps)]

# Build a(L, rho) and da(L, rho)
L = np.array(widths, dtype=float)
rho = np.array(ps, dtype=float)
a = np.zeros((len(L), len(rho)))
da = np.zeros_like(a)
for i, w in enumerate(widths):
    sub = summary[summary["mlp_width"] == w].set_index("p")
    for j, p in enumerate(ps):
        a[i, j] = float(sub.loc[p, "mean_acc"])
        da[i, j] = float(sub.loc[p, "stderr"])

# Initial guesses
rho_c0 = float(np.median(rho))
nu0 = 1.0
zeta0 = 0.0

ret = fssa.autoscale(L, rho, a, da, rho_c0, nu0, zeta0)
print("autoscale result:")
print(ret)

scaled = fssa.scaledata(L, rho, a, da, ret.rho, ret.nu, ret.zeta)
quality = fssa.quality(scaled.x, scaled.y, scaled.dy)
print(f"collapse quality S (reduced chi^2): {quality:.4f}")

# Plot collapsed data
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
for i, w in enumerate(widths):
    ax.plot(scaled.x[i], scaled.y[i], marker="o", linestyle="", label=f"w={w}")
ax.set_title(f"pyfssa collapse (S={quality:.3f})")
ax.set_xlabel("scaled x")
ax.set_ylabel("scaled y")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()



## 50/50 split, train on 1% subset (no noise), width=64 depth=1


In [ ]:
# 50/50 split, train on 1% subset (no noise), width=64 depth=1
# Activations: relu, tanh, linear, quadratic; 10 repeats with random subset each time
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.nnet_models import MLP, train_one_epoch, evaluate

base_seed = 1234
rng = np.random.default_rng(base_seed)

torch.manual_seed(base_seed)

transform = transforms.Compose([
    transforms.ToTensor(),
])

# Use MNIST train split and create a 50/50 train-test split
full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_size = len(full) // 2
test_size = len(full) - train_size
generator = torch.Generator().manual_seed(base_seed)
train_base, test_base = torch.utils.data.random_split(full, [train_size, test_size], generator=generator)

activations = ["relu", "tanh", "linear", "quadratic"]
epochs = 20
repeats = 10

all_accs = {act: [] for act in activations}

for rep in range(repeats):
    rep_seed = base_seed + rep
    rng = np.random.default_rng(rep_seed)
    torch.manual_seed(rep_seed)

    # Keep only 1% of the training split (random subset per repeat)
    train_indices = np.arange(len(train_base))
    rng.shuffle(train_indices)
    keep = max(1, int(0.01 * len(train_base)))
    train_subset = Subset(train_base, train_indices[:keep].tolist())

    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_base, batch_size=256, shuffle=False, num_workers=2)

    for act in activations:
        model = MLP(act, hidden_sizes=[64])
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        test_accs = []
        for _ in tqdm(range(epochs), desc=f"rep {rep+1}/{repeats} {act}", unit="epoch"):
            train_one_epoch(model, train_loader, optimizer, torch.device("cpu"), loss_type="cross_entropy")
            _, acc = evaluate(model, test_loader, torch.device("cpu"), loss_type="cross_entropy")
            test_accs.append(acc)
        all_accs[act].append(test_accs)

# Compute mean and stderr across repeats per activation and epoch
plt.figure(figsize=(12, 8))
for act in activations:
    arr = np.array(all_accs[act])  # shape (repeats, epochs)
    mean_acc = arr.mean(axis=0)
    stderr = arr.std(axis=0, ddof=1) / np.sqrt(repeats)
    epochs_axis = np.arange(1, epochs + 1)
    plt.errorbar(epochs_axis, mean_acc, yerr=stderr, marker="o", capsize=2, label=act)

plt.xlabel("epoch")
plt.ylabel("test accuracy")
plt.title("50/50 split, 1% train subset (w=64, d=1) | Mean ± stderr over 10 repeats")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1) noisy complementary set results
pristine_path = 'results/data/results_per_run_mlp_r50_e10_pristine0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_pristine_fraction.csv'
noisy = pd.read_csv(pristine_path)

# 2) clean subset benchmarks
bench_path = 'results/data/results_per_run_mlp_r100_e20_tf0.500_clean0.001-1.000_w64_d1_loss-cross_entropy_clean_subset_benchmarks.csv'
bench = pd.read_csv(bench_path)

# Summaries
noisy_sum = (
    noisy.groupby(['activation', 'pristine_frac', 'p'], as_index=False)
    .agg(mean_test_accuracy=('test_accuracy', 'mean'))
)
bench_sum = (
    bench.groupby(['activation', 'clean_frac'], as_index=False)
    .agg(mean_test_accuracy=('test_accuracy', 'mean'))
)

# Skip pristine_frac = 0 and clean_frac = 0.001
pristine_fracs = sorted([f for f in noisy_sum['pristine_frac'].unique() if f > 0])
clean_fracs = sorted([f for f in bench_sum['clean_frac'].unique() if abs(f - 0.001) > 1e-12])

# Ensure we can match by fraction value
clean_lookup = {
    (row['activation'], row['clean_frac']): row['mean_test_accuracy']
    for _, row in bench_sum.iterrows()
}

activations = sorted(noisy_sum['activation'].unique())

nrows = len(pristine_fracs)
ncols = len(activations)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=True)
if nrows == 1:
    axes = [axes]

for r, frac in enumerate(pristine_fracs):
    for c, act in enumerate(activations):
        ax = axes[r][c] if ncols > 1 else axes[r]
        d = noisy_sum[(noisy_sum['activation'] == act) & (noisy_sum['pristine_frac'] == frac)].sort_values('p')
        ax.plot(d['p'], d['mean_test_accuracy'], marker='o', label='clean subset + noisy complement')

        # Benchmark line (match on same fraction)
        bench_y = clean_lookup.get((act, frac))
        if bench_y is not None:
            ax.axhline(bench_y, color='black', linestyle='--', label='clean training subset benchmark')

        ax.set_title(f'{act} / frac={frac:.3f}')
        ax.set_xlabel('p')
        if c == 0:
            ax.set_ylabel('mean test accuracy')
        ax.grid(True, alpha=0.3)

# One legend for the whole figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=True)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1) noisy complementary set results
pristine_path = 'results/data/results_per_run_mlp_r50_e10_pristine0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_pristine_fraction.csv'
noisy = pd.read_csv(pristine_path)

# 2) clean subset benchmarks
bench_path = 'results/data/results_per_run_mlp_r100_e20_tf0.500_clean0.001-1.000_w64_d1_loss-cross_entropy_clean_subset_benchmarks.csv'
bench = pd.read_csv(bench_path)

# Summaries
noisy_sum = (
    noisy.groupby(['activation', 'pristine_frac', 'p'], as_index=False)
    .agg(mean_test_accuracy=('test_accuracy', 'mean'))
)
bench_sum = (
    bench.groupby(['activation', 'clean_frac'], as_index=False)
    .agg(mean_test_accuracy=('test_accuracy', 'mean'))
)

# Skip pristine_frac = 0 and 0.02
pristine_fracs = sorted([f for f in noisy_sum['pristine_frac'].unique() if f > 0 and abs(f - 0.02) > 1e-12])
clean_lookup = {
    (row['activation'], row['clean_frac']): row['mean_test_accuracy']
    for _, row in bench_sum.iterrows()
}

activations = sorted(noisy_sum['activation'].unique())

nrows = len(pristine_fracs)
ncols = len(activations)
fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 6 * nrows), sharex=True, sharey=True)

if nrows == 1:
    axes = [axes]

for r, frac in enumerate(pristine_fracs):
    for c, act in enumerate(activations):
        ax = axes[r][c] if ncols > 1 else axes[r]
        d = noisy_sum[(noisy_sum['activation'] == act) & (noisy_sum['pristine_frac'] == frac)].sort_values('p')
        ax.plot(d['p'], d['mean_test_accuracy'], marker='o', label='clean training subset + noisy complimentary set')

        # Benchmark line (match on same fraction)
        bench_y = clean_lookup.get((act, frac))
        if bench_y is not None:
            ax.axhline(bench_y, color='black', linestyle='--', label='clean training subset benchmark')

        ax.set_title(f'{act} / frac={frac:.3f}')
        ax.set_xlabel('p')  # x-label on every subplot
        ax.set_ylabel('mean test accuracy' if c == 0 else "")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=12, location='lower left')

out_path = Path('results/figures') / 'pristine_vs_clean_benchmark_grid.png'
fig.savefig(out_path, dpi=200)
print(f'Saved: {out_path}')
plt.show()



In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_rmax100_e20_tf0.500_p0.00-1.00_w64-64_d1_loss-cross_entropy_width_sweep_relu_only.csv'
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

model_types = sorted(summary['model_type'].unique())
for model_type in model_types:
    sub = summary[summary['model_type'] == model_type]
    activations = sorted(sub['activation'].unique())
    widths = sorted(sub['mlp_width'].dropna().unique())
    n = len(activations)

    # Mean test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_accuracy'],
                yerr=d['stderr_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Std test accuracy
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['std_test_accuracy'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('std test accuracy')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean test loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.errorbar(
                d['p'],
                d['mean_test_loss'],
                yerr=d['stderr_test_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean test loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Mean train loss
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
    if n == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        d_act = sub[sub['activation'] == act]
        for w in widths:
            d = d_act[d_act['mlp_width'] == w].sort_values('p')
            ax.plot(
                d['p'],
                d['mean_train_loss'],
                marker='o',
                label=f'w={w}',
            )
        ax.set_title(f'{model_type} / {act}')
        ax.set_xlabel('p')
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel('mean train loss')
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

per_run_path = Path('results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv')
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

cache_key = per_run_path.name.replace('results_per_run_', '').replace('.csv', '')
out_pdf = per_run_path.with_name(f'plots_{cache_key}.pdf')

with PdfPages(out_pdf) as pdf:
    model_types = sorted(summary['model_type'].unique())
    for model_type in model_types:
        sub = summary[summary['model_type'] == model_type]
        activations = sorted(sub['activation'].unique())
        widths = sorted(sub['mlp_width'].dropna().unique())
        n = len(activations)

        # Mean test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(
                    d['p'],
                    d['mean_test_accuracy'],
                    yerr=d['stderr_test_accuracy'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Std test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(
                    d['p'],
                    d['std_test_accuracy'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('std test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Mean test loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(
                    d['p'],
                    d['mean_test_loss'],
                    yerr=d['stderr_test_loss'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

        # Mean train loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(
                    d['p'],
                    d['mean_train_loss'],
                    marker='o',
                    label=f'w={w}',
                )
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean train loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f'Saved: {out_pdf}')


In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

per_run_path = Path('results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv')
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'model_type', 'corruption_mode', 'p', 'sigma', 'mlp_width'], as_index=False)
    .agg(
        repeats=('test_accuracy', 'size'),
        mean_test_accuracy=('test_accuracy', 'mean'),
        std_test_accuracy=('test_accuracy', 'std'),
        mean_test_loss=('test_loss', 'mean'),
        std_test_loss=('test_loss', 'std'),
        mean_train_loss=('train_loss', 'mean'),
        std_train_loss=('train_loss', 'std'),
    )
)
summary['stderr_test_accuracy'] = summary['std_test_accuracy'] / summary['repeats'].pow(0.5)
summary['stderr_test_loss'] = summary['std_test_loss'] / summary['repeats'].pow(0.5)

cache_key = per_run_path.name.replace('results_per_run_', '').replace('.csv', '')
out_pdf = per_run_path.with_name(f'plots_{cache_key}.pdf')

with PdfPages(out_pdf) as pdf:
    model_types = sorted(summary['model_type'].unique())
    for model_type in model_types:
        sub = summary[summary['model_type'] == model_type]
        activations = sorted(sub['activation'].unique())
        widths = sorted(sub['mlp_width'].dropna().unique())
        n = len(activations)

        # Mean test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(d['p'], d['mean_test_accuracy'], yerr=d['stderr_test_accuracy'], marker='o', label=f'w={w}')
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()
        plt.close(fig)

        # Std test accuracy
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(d['p'], d['std_test_accuracy'], marker='o', label=f'w={w}')
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('std test accuracy')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()
        plt.close(fig)

        # Mean test loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.errorbar(d['p'], d['mean_test_loss'], yerr=d['stderr_test_loss'], marker='o', label=f'w={w}')
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean test loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()
        plt.close(fig)

        # Mean train loss
        fig, axes = plt.subplots(1, n, figsize=(4 * n, 3), sharey=True)
        if n == 1:
            axes = [axes]
        for ax, act in zip(axes, activations):
            d_act = sub[sub['activation'] == act]
            for w in widths:
                d = d_act[d_act['mlp_width'] == w].sort_values('p')
                ax.plot(d['p'], d['mean_train_loss'], marker='o', label=f'w={w}')
            ax.set_title(f'{model_type} / {act}')
            ax.set_xlabel('p')
            ax.grid(True, alpha=0.3)
        axes[0].set_ylabel('mean train loss')
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.show()
        plt.close(fig)

print(f'Saved: {out_pdf}')


In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv'
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

summary = (
    rows.groupby(['activation', 'mlp_width', 'p'], as_index=False)
    .agg(mean_test_accuracy=('test_accuracy', 'mean'))
)

# Build mean/stderr for S
stats = (
    rows.groupby(['mlp_width', 'p'], as_index=False)['test_accuracy']
    .agg(['mean', 'std', 'count']).reset_index()
    .rename(columns={'mean': 'mean_acc', 'std': 'std_acc', 'count': 'n'})
)
stats['stderr'] = stats['std_acc'] / np.sqrt(stats['n'])

min_nonzero = stats.loc[stats['stderr'] > 0, 'stderr'].min()
stats['stderr'] = stats['stderr'].fillna(min_nonzero if np.isfinite(min_nonzero) else 1e-6)
stats['stderr'] = stats['stderr'].clip(lower=1e-6)

# Finite-size scaling collapse (grid search)
def collapse_objective(df, pc, nu):
    widths = sorted(df['mlp_width'].dropna().unique())
    curves = {}
    xs_all = []
    for w in widths:
        d = df[df['mlp_width'] == w].sort_values('p')
        p = d['p'].values
        s = d['mean_test_accuracy'].values
        if pc < p.min() or pc > p.max():
            return np.inf
        s_pc = np.interp(pc, p, s)
        x = (p - pc) * (w ** (1.0 / nu))
        y = s - s_pc
        curves[w] = (x, y)
        xs_all.append(x)
    if not curves:
        return np.inf
    xs_all = np.unique(np.concatenate(xs_all))
    total = 0.0
    count = 0
    for x0 in xs_all:
        ys = []
        for x_vals, y_vals in curves.values():
            if x0 < x_vals.min() or x0 > x_vals.max():
                continue
            ys.append(np.interp(x0, x_vals, y_vals))
        if len(ys) < 2:
            continue
        ybar = float(np.mean(ys))
        total += float(np.sum((np.array(ys) - ybar) ** 2))
        count += len(ys)
    return total, count

def collapse_quality(df, pc, nu):
    total, count = collapse_objective(df, pc, nu)
    return np.inf if count == 0 else total / count  # SSE per usable point

def collapse_quality_S(stats_df, pc, nu):
    widths = sorted(stats_df['mlp_width'].dropna().unique())
    curves = {}
    xs_all = []
    for w in widths:
        d = stats_df[stats_df['mlp_width'] == w].sort_values('p')
        p = d['p'].values
        s = d['mean_acc'].values
        e = d['stderr'].values
        if pc < p.min() or pc > p.max():
            return np.inf
        s_pc = np.interp(pc, p, s)
        x = (p - pc) * (w ** (1.0 / nu))
        y = s - s_pc
        curves[w] = (x, y, e)
        xs_all.append(x)
    xs_all = np.unique(np.concatenate(xs_all))

    chi2 = 0.0
    dof = 0
    for x0 in xs_all:
        ys = []
        es = []
        for x_vals, y_vals, e_vals in curves.values():
            if x0 < x_vals.min() or x0 > x_vals.max():
                continue
            ys.append(np.interp(x0, x_vals, y_vals))
            es.append(np.interp(x0, x_vals, e_vals))
        if len(ys) < 2:
            continue
        ys = np.array(ys)
        es = np.array(es)
        wts = 1.0 / (es**2)
        ybar = np.sum(wts * ys) / np.sum(wts)
        chi2 += np.sum(((ys - ybar) / es) ** 2)
        dof += len(ys) - 1

    return chi2 / max(dof, 1)

act = 'relu'
sub = summary[summary['activation'] == act]

pc_grid = np.linspace(sub['p'].min() + 1e-3, sub['p'].max() - 1e-3, 40)
nu_grid = np.linspace(0.2, 5.0, 40)

best = (np.inf, None, None)
for pc in pc_grid:
    for nu in nu_grid:
        q = collapse_quality(sub, pc, nu)
        if q < best[0]:
            best = (q, pc, nu)

quality, pc_star, nu_star = best
S = collapse_quality_S(stats, pc_star, nu_star)

print(f"relu collapse: pc*={pc_star:.4f}, nu*={nu_star:.3f}, Q={quality:.6e}, S={S:.4f}")

# Plot collapse
plt.figure(figsize=(6, 4))
for w in sorted(sub['mlp_width'].dropna().unique()):
    d = sub[sub['mlp_width'] == w].sort_values('p')
    p = d['p'].values
    s = d['mean_test_accuracy'].values
    s_pc = np.interp(pc_star, p, s)
    x = (p - pc_star) * (w ** (1.0 / nu_star))
    y = s - s_pc
    plt.plot(x, y, marker='.', linestyle='', label=f'w={w}')
plt.title(f"Collapse: relu (pc*={pc_star:.3f}, nu*={nu_star:.2f}, Q={quality:.2e}, S={S:.3f})")
plt.xlabel(r"$(p - p^*) w^{1/nu}$")
plt.ylabel(r"$\mathrm{mean_test_acc}(p) - \mathrm{mean_test_acc}(p^*)$")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()

plt.savefig('relu_collapse.pdf', dpi=200, bbox_inches='tight')
plt.show()



## pyfssa (fssa) finite-size scaling with zeta fixed to 0


In [ ]:
# pyfssa (fssa) finite-size scaling with zeta fixed to 0
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

# --- data ---
per_run_path = "results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv"
raw = pd.read_csv(per_run_path)
raw = raw[(raw["activation"] == "relu") & (raw["model_type"] == "mlp")].copy()

# Parse width from serialized list, e.g. "[128]"
raw["mlp_width"] = raw["mlp_hidden_sizes"].apply(
    lambda s: int(ast.literal_eval(s)[0]) if isinstance(s, str) else int(s[0])
)

summary = (
    raw.groupby(["mlp_width", "p"], as_index=False)["test_accuracy"]
    .agg(["mean", "std", "count"]).reset_index()
    .rename(columns={"mean": "mean_acc", "std": "std_acc", "count": "n"})
)
summary["stderr"] = summary["std_acc"] / np.sqrt(summary["n"])

# Optional: restrict p-window to improve collapse
p_min, p_max = 0.90, 0.98
summary = summary[(summary["p"] >= p_min) & (summary["p"] <= p_max)].copy()

# Ensure strictly positive errors for fssa
min_nonzero = summary.loc[summary["stderr"] > 0, "stderr"].min()
summary["stderr"] = summary["stderr"].fillna(min_nonzero if np.isfinite(min_nonzero) else 1e-6)
summary["stderr"] = summary["stderr"].clip(lower=1e-6)

# Use only p-values present for all widths
widths = sorted(summary["mlp_width"].unique())
common_p = None
for w in widths:
    pset = set(summary.loc[summary["mlp_width"] == w, "p"].values)
    common_p = pset if common_p is None else (common_p & pset)
ps = sorted(common_p) if common_p else sorted(summary["p"].unique())
summary = summary[summary["p"].isin(ps)]

# Build a(L, rho) and da(L, rho)
L = np.array(widths, dtype=float)
rho = np.array(ps, dtype=float)
a = np.zeros((len(L), len(rho)))
da = np.zeros_like(a)
for i, w in enumerate(widths):
    sub = summary[summary["mlp_width"] == w].set_index("p")
    for j, p in enumerate(ps):
        a[i, j] = float(sub.loc[p, "mean_acc"])
        da[i, j] = float(sub.loc[p, "stderr"])

# Fix zeta = 0 and grid-search for rho_c, nu
zeta_fixed = 0.0
rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 40)
nu_grid = np.linspace(0.2, 5.0, 40)

best = (np.inf, None, None)
for rho_c in rho_grid:
    for nu in nu_grid:
        scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
        q = fssa.quality(scaled.x, scaled.y, scaled.dy)
        if q < best[0]:
            best = (q, rho_c, nu)

quality, rho_c, nu = best
print(f"fixed zeta=0: rho_c={rho_c:.4f}, nu={nu:.3f}, S={quality:.4f}")

scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)

# Plot collapsed data
fig, ax = plt.subplots(figsize=(8, 6))
for i, w in enumerate(widths):
    ax.plot(scaled.x[i], scaled.y[i], marker=".", linestyle="", label=f"w={w}")
ax.set_title(f"pyfssa collapse (S={quality:.3f})")
ax.set_xlabel(rf"$x=(p-p^*)L^{{1/\nu}}$, $p^*={rho_c:.4f}$, $\nu={nu:.3f}$")
ax.set_ylabel(r"$\mathrm{(mean \ test \ acc)}(p) - \mathrm{(mean \ test \ acc)}(p^*)$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=12)
plt.tight_layout()

out_pdf = "relu_fssa_collapse_zeta0.pdf"
plt.savefig(out_pdf, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {out_pdf}")



In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv'
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

# std_test_accuracy vs p for each width (relu only)
summary = (
    rows[(rows['activation'] == 'relu') & (rows['model_type'] == 'mlp')]
    .groupby(['mlp_width', 'p'], as_index=False)
    .agg(std_test_accuracy=('test_accuracy', 'std'))
)

widths = sorted(summary['mlp_width'].dropna().unique())

plt.figure(figsize=(5, 4))
for w in widths:
    d = summary[summary['mlp_width'] == w].sort_values('p')
    plt.plot(d['p'], d['std_test_accuracy'], marker='o', label=f'w={w}')
plt.xlabel('p')
plt.ylabel('std test accuracy')
plt.title('std(test accuracy) vs p (relu)')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Peak location and height vs width
peak_rows = []
for w in widths:
    d = summary[summary['mlp_width'] == w].sort_values('p')
    if d['std_test_accuracy'].isna().all():
        continue
    idx = d['std_test_accuracy'].idxmax()
    peak_rows.append((w, float(d.loc[idx, 'p']), float(d.loc[idx, 'std_test_accuracy'])))

if peak_rows:
    widths_arr = np.array([r[0] for r in peak_rows])
    p_peaks = np.array([r[1] for r in peak_rows])
    std_peaks = np.array([r[2] for r in peak_rows])

    fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
    ax[0].plot(widths_arr, p_peaks, marker='o')
    ax[0].set_xlabel('width')
    ax[0].set_ylabel('p at peak std')
    ax[0].set_title('Peak location vs width')
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(widths_arr, std_peaks, marker='o')
    ax[1].set_xlabel('width')
    ax[1].set_ylabel('peak std')
    ax[1].set_title('Peak height vs width')
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

per_run_path = 'results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv'
rows = pd.read_csv(per_run_path)

# Recover width if missing
if 'mlp_width' not in rows.columns:
    def parse_width(value: str) -> int:
        try:
            sizes = ast.literal_eval(value)
            if isinstance(sizes, list) and sizes:
                return int(sizes[0])
        except Exception:
            pass
        return None
    rows['mlp_width'] = rows['mlp_hidden_sizes'].apply(parse_width)

# mean_test_loss vs p for each width (relu only)
summary = (
    rows[(rows['activation'] == 'relu') & (rows['model_type'] == 'mlp')]
    .groupby(['mlp_width', 'p'], as_index=False)
    .agg(mean_test_loss=('test_loss', 'mean'))
)

widths = sorted(summary['mlp_width'].dropna().unique())

plt.figure(figsize=(5, 4))
for w in widths:
    d = summary[summary['mlp_width'] == w].sort_values('p')
    plt.plot(d['p'], d['mean_test_loss'], marker='o', label=f'w={w}')
plt.xlabel('p')
plt.ylabel('mean test loss')
plt.title('mean test loss vs p (relu)')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Peak location and height vs width (max mean loss)
peak_rows = []
for w in widths:
    d = summary[summary['mlp_width'] == w].sort_values('p')
    if d['mean_test_loss'].isna().all():
        continue
    idx = d['mean_test_loss'].idxmax()
    peak_rows.append((w, float(d.loc[idx, 'p']), float(d.loc[idx, 'mean_test_loss'])))

if peak_rows:
    widths_arr = np.array([r[0] for r in peak_rows])
    p_peaks = np.array([r[1] for r in peak_rows])
    loss_peaks = np.array([r[2] for r in peak_rows])

    fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
    ax[0].plot(widths_arr, p_peaks, marker='o')
    ax[0].set_xlabel('width')
    ax[0].set_ylabel('p at peak loss')
    ax[0].set_title('Peak location vs width')
    ax[0].grid(True, alpha=0.3)

    ax[1].plot(widths_arr, loss_peaks, marker='o')
    ax[1].set_xlabel('width')
    ax[1].set_ylabel('peak mean loss')
    ax[1].set_title('Peak height vs width')
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


## clean_frac_noisy_vs_benchmark: reproduce script plots from summary CSV


In [ ]:
# clean_frac_noisy_vs_benchmark: reproduce script plots from summary CSV
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

summary_path = Path('results/data/results_summary_mlp_rn10_rc10_rfc1_e20_tf0.500_clean0.001-0.100_p0.00-1.00_w64_d1_loss-cross_entropy_clean_frac_noisy_vs_benchmark.csv')
df = pd.read_csv(summary_path)

noisy = df[df['mode'] == 'noisy_complement'].copy()
bench = df[df['mode'] == 'clean_only'].copy()
full_clean = df[df['mode'] == 'full_clean_once'].copy()

if noisy.empty:
    raise ValueError('No rows with mode=noisy_complement were found in the summary CSV.')

# Locked visual encoding (matches script)
color_noisy = '#1f77b4'
color_clean_subset = '#000000'
color_full_clean = '#666666'

activations = sorted(noisy['activation'].unique())
clean_fracs = sorted([f for f in noisy['clean_frac'].unique() if f > 0])
if not clean_fracs:
    clean_fracs = sorted(noisy['clean_frac'].unique())

bench_lookup = {
    (row['activation'], row['clean_frac']): row['mean_test_accuracy']
    for _, row in bench.iterrows()
}
full_clean_lookup = {
    row['activation']: row['mean_test_accuracy']
    for _, row in full_clean.iterrows()
}

nrows = len(clean_fracs)
ncols = len(activations)
fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(4 * ncols, 3 * nrows),
    sharex=True,
    sharey=True,
)
if nrows == 1:
    axes = [axes]

for r, frac in enumerate(clean_fracs):
    for c, act in enumerate(activations):
        ax = axes[r][c] if ncols > 1 else axes[r]
        d = noisy[(noisy['activation'] == act) & (noisy['clean_frac'] == frac)].sort_values('p')
        if d.empty:
            continue

        ax.errorbar(
            d['p'],
            d['mean_test_accuracy'],
            yerr=d['stderr_test_accuracy'],
            marker='o',
            color=color_noisy,
            label='clean training subset + noisy complementary set',
        )

        bench_y = bench_lookup.get((act, frac))
        if bench_y is not None:
            ax.axhline(
                bench_y,
                color=color_clean_subset,
                linestyle='--',
                linewidth=1.5,
                label='clean training subset benchmark',
            )

        full_clean_y = full_clean_lookup.get(act)
        if full_clean_y is not None:
            ax.axhline(
                full_clean_y,
                color=color_full_clean,
                linestyle=':',
                linewidth=1.8,
                label='entired clean dataset',
            )

        ax.set_title(f'{act} / frac={frac:.3f}')
        ax.set_xlabel('p')
        if c == 0:
            ax.set_ylabel('mean test accuracy')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

fig.suptitle('Clean subset + noisy complementary set vs clean-only benchmark', y=1.02)
plt.tight_layout()

out_png = Path('results/figures') / 'comparison_grid_mlp_rn10_rc10_rfc1_e20_tf0.500_clean0.001-0.100_p0.00-1.00_w64_d1_loss-cross_entropy_clean_frac_noisy_vs_benchmark.png'
out_pdf = Path('results/figures') / 'comparison_grid_mlp_rn10_rc10_rfc1_e20_tf0.500_clean0.001-0.100_p0.00-1.00_w64_d1_loss-cross_entropy_clean_frac_noisy_vs_benchmark.pdf'
fig.savefig(out_png, dpi=200, bbox_inches='tight')
fig.savefig(out_pdf, dpi=200, bbox_inches='tight')
print(f'Saved: {out_png}')
print(f'Saved: {out_pdf}')
plt.show()




## Infinite-width ReLU NTK: mean test accuracy vs corruption probability p (100 realizations)


In [ ]:
# Infinite-width ReLU NTK: mean test accuracy vs corruption probability p (100 realizations)
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import jax
import jax.numpy as jnp
import neural_tangents as nt
from neural_tangents import stax

import torch
from torchvision import datasets, transforms

# ----------------------------
# Config (edit as needed)
# ----------------------------
seed = 1234
num_realizations = 100
p_values = np.linspace(0.0, 1.0, 21)

# Subset sizes are tunable; NTK scales ~O(n_train^3)
n_train = 512
n_test = 512

ridge = 1e-6  # Tikhonov regularization for kernel solve
batch_kernel = False  # set True if you later want to write a batched kernel helper

rng = np.random.default_rng(seed)

# ----------------------------
# Data
# ----------------------------
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root='data', train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root='data', train=False, download=True, transform=transform)

train_idx = rng.choice(len(train_ds), size=n_train, replace=False)
test_idx = rng.choice(len(test_ds), size=n_test, replace=False)

x_train = np.stack([train_ds[i][0].numpy().reshape(-1) for i in train_idx]).astype(np.float32)
y_train = np.array([train_ds[i][1] for i in train_idx], dtype=np.int32)
x_test = np.stack([test_ds[i][0].numpy().reshape(-1) for i in test_idx]).astype(np.float32)
y_test = np.array([test_ds[i][1] for i in test_idx], dtype=np.int32)

# One-hot targets for kernel regression
Y_train = np.eye(10, dtype=np.float32)[y_train]  # [n_train, 10]

# ----------------------------
# Infinite-width ReLU NTK kernel
# ----------------------------
# One hidden ReLU block; infinite-width limit is represented by kernel_fn.
_, _, kernel_fn = stax.serial(
    stax.Dense(1024, W_std=1.0, b_std=0.0),
    stax.Relu(),
    stax.Dense(1, W_std=1.0, b_std=0.0),
)

def corrupt_replacement(x: np.ndarray, p: float, rng_local: np.random.Generator) -> np.ndarray:
    """Replacement channel: each pixel replaced by U[0,1] with prob p."""
    mask = rng_local.random(x.shape, dtype=np.float32) < p
    repl = rng_local.random(x.shape, dtype=np.float32)
    return np.where(mask, repl, x).astype(np.float32)

def ntk_predict_scores(x_train_corr: np.ndarray, Y: np.ndarray, x_test_clean: np.ndarray, lam: float) -> np.ndarray:
    """Kernel ridge on infinite-width NTK; returns class scores [n_test, 10]."""
    K_tt = np.asarray(kernel_fn(jnp.asarray(x_train_corr), None, get='ntk'), dtype=np.float64)
    K_xt = np.asarray(kernel_fn(jnp.asarray(x_test_clean), jnp.asarray(x_train_corr), get='ntk'), dtype=np.float64)

    n = K_tt.shape[0]
    # Scale-invariant regularization improves conditioning across p.
    reg = lam * (np.trace(K_tt) / max(n, 1))
    A = K_tt + reg * np.eye(n, dtype=np.float64)

    alpha = np.linalg.solve(A, Y.astype(np.float64))      # [n_train, 10]
    scores = K_xt @ alpha                                  # [n_test, 10]
    return scores

# ----------------------------
# Sweep p and average over realizations
# ----------------------------
rows = []
for p in tqdm(p_values, desc='p sweep'):
    accs = []
    for _ in range(num_realizations):
        local_seed = int(rng.integers(1, 1_000_000_000))
        local_rng = np.random.default_rng(local_seed)

        x_train_corr = corrupt_replacement(x_train, float(p), local_rng)
        scores = ntk_predict_scores(x_train_corr, Y_train, x_test, ridge)
        y_pred = np.argmax(scores, axis=1)
        acc = float(np.mean(y_pred == y_test))
        accs.append(acc)

    rows.append(
        {
            'p': float(p),
            'repeats': int(num_realizations),
            'mean_test_accuracy': float(np.mean(accs)),
            'std_test_accuracy': float(np.std(accs, ddof=0)),
            'stderr_test_accuracy': float(np.std(accs, ddof=0) / np.sqrt(len(accs))),
        }
    )

# ----------------------------
# Plot
# ----------------------------
p_arr = np.array([r['p'] for r in rows])
mean_arr = np.array([r['mean_test_accuracy'] for r in rows])
se_arr = np.array([r['stderr_test_accuracy'] for r in rows])

plt.figure(figsize=(6, 4))
plt.errorbar(p_arr, mean_arr, yerr=se_arr, marker='o', color='#1f77b4')
plt.xlabel('corruption probability p')
plt.ylabel('mean test accuracy (clean test set)')
plt.title(f'Infinite-width ReLU NTK (n_train={n_train}, n_test={n_test}, repeats={num_realizations})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Optional: inspect as a dataframe
import pandas as pd
ntk_summary_df = pd.DataFrame(rows)
ntk_summary_df




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

summary_path = Path('results/data/results_summary_ntk_relu_r100_na_ntr512_nte512_p0.75-1.00_ridge1.0e-06_infinite_width_relu_ntk.csv')
if not summary_path.exists():
    raise FileNotFoundError(f'Missing file: {summary_path}')

summary = pd.read_csv(summary_path).sort_values('p')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].errorbar(
    summary['p'],
    summary['mean_test_accuracy'],
    yerr=summary['stderr_test_accuracy'],
    marker='o',
    linestyle='-',
    color='#1f77b4',
)
axes[0].set_title('Mean Test Accuracy vs p')
axes[0].set_xlabel('corruption probability p')
axes[0].set_ylabel('mean test accuracy')
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    summary['p'],
    summary['std_test_accuracy'],
    marker='o',
    linestyle='-',
    color='#d62728',
)
axes[1].set_title('Std Test Accuracy vs p')
axes[1].set_xlabel('corruption probability p')
axes[1].set_ylabel('std test accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## pyfssa collapse on infimnist n_train sweep (zeta fixed to 0, n_train >= 10000)


In [ ]:
# pyfssa collapse on infimnist n_train sweep (zeta fixed to 0, n_train >= 10000)
import csv
import numpy as np
import matplotlib.pyplot as plt
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

per_run_path = "results/data/results_per_run_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr100-1000000_nte100-1000000_p0.75-1.00_w256_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

# Read only columns needed for collapse to avoid pandas dependency.
rows = []
with open(per_run_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "test_accuracy", "n_train_samples"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            acc = float(r["test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, acc))

if not rows:
    raise ValueError("No valid rows found in per-run CSV.")

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
acc_all = arr[:, 2]

# Keep only n_train >= 10000
mask = n_train_all >= 10000
n_train_all = n_train_all[mask]
p_all = p_all[mask]
acc_all = acc_all[mask]
if n_train_all.size == 0:
    raise ValueError("No rows remain after filtering to n_train_samples >= 10000.")

n_sizes = np.array(sorted(np.unique(n_train_all)), dtype=int)
if n_sizes.size < 2:
    raise ValueError(f"Need >=2 n_train values for collapse, found {n_sizes.tolist()}")

# Keep only p values that exist for all n_train sizes
p_common = None
for ntr in n_sizes:
    pset = set(np.unique(p_all[n_train_all == ntr]).tolist())
    p_common = pset if p_common is None else (p_common & pset)
ps = np.array(sorted(p_common), dtype=float)
if ps.size < 2:
    raise ValueError("Need >=2 shared p values across n_train sizes.")

# Build a(L, rho) = mean accuracy and da(L, rho) = stderr
L = n_sizes.astype(float)  # system size = n_train
rho = ps
a = np.zeros((L.size, rho.size), dtype=float)
da = np.zeros_like(a)

for i, ntr in enumerate(n_sizes):
    for j, p in enumerate(ps):
        m = (n_train_all == ntr) & np.isclose(p_all, p)
        vals = acc_all[m]
        if vals.size == 0:
            raise ValueError(f"Missing data for n_train={ntr}, p={p}")
        a[i, j] = float(np.mean(vals))
        std = float(np.std(vals, ddof=0)) if vals.size > 1 else 0.0
        da[i, j] = std / np.sqrt(vals.size)

# Ensure strictly positive errors for fssa
nonzero = da[da > 0]
min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

# Fixed zeta = 0; grid-search rho_c and nu
zeta_fixed = 0.0
rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 40)
nu_grid = np.linspace(0.2, 5.0, 40)

best = (np.inf, None, None)
for rho_c in rho_grid:
    for nu in nu_grid:
        scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
        S = fssa.quality(scaled.x, scaled.y, scaled.dy)
        if S < best[0]:
            best = (S, rho_c, nu)

S_star, rho_c_star, nu_star = best
print(f"fixed zeta=0: rho_c={rho_c_star:.4f}, nu={nu_star:.3f}, S={S_star:.4f}")

scaled = fssa.scaledata(L, rho, a, da, rho_c_star, nu_star, zeta_fixed)

fig, ax = plt.subplots(figsize=(8, 6))
for i, ntr in enumerate(n_sizes):
    ax.plot(scaled.x[i], scaled.y[i], marker=".", linestyle="", label=f"n_train={ntr}")
ax.set_title(f"pyfssa collapse (zeta=0, S={S_star:.3f})")
ax.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={rho_c_star:.4f}$, $\nu={nu_star:.3f}$")
ax.set_ylabel(r"$\mathrm{mean\ test\ acc}(p)-\mathrm{mean\ test\ acc}(p^*)$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()




## pyfssa collapse on infimnist n_train sweep (summary CSV, zeta fixed to 0)


In [ ]:
# pyfssa collapse on infimnist n_train sweep (summary CSV, zeta fixed to 0)
import csv
import numpy as np
import matplotlib.pyplot as plt
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy", "stderr_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            mean_acc = float(r["mean_test_accuracy"])
            stderr = float(r["stderr_test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, mean_acc, stderr))

if not rows:
    raise ValueError("No valid rows found in summary CSV.")

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
mean_all = arr[:, 2]
stderr_all = arr[:, 3]

# Keep only n_train >= 10000 (expected for this summary)
mask = n_train_all >= 10000
n_train_all = n_train_all[mask]
p_all = p_all[mask]
mean_all = mean_all[mask]
stderr_all = stderr_all[mask]
if n_train_all.size == 0:
    raise ValueError("No rows remain after filtering to n_train_samples >= 10000.")

n_sizes = np.array(sorted(np.unique(n_train_all)), dtype=int)
if n_sizes.size < 2:
    raise ValueError(f"Need >=2 n_train values for collapse, found {n_sizes.tolist()}")

# Keep only p values that exist for all n_train sizes
p_common = None
for ntr in n_sizes:
    pset = set(np.unique(p_all[n_train_all == ntr]).tolist())
    p_common = pset if p_common is None else (p_common & pset)
ps = np.array(sorted(p_common), dtype=float)
if ps.size < 2:
    raise ValueError("Need >=2 shared p values across n_train sizes.")

# Build a(L, rho) and da(L, rho)
L = n_sizes.astype(float)  # system size = n_train
rho = ps
nL, nR = L.size, rho.size
a = np.zeros((nL, nR), dtype=float)
da = np.zeros_like(a)

for i, ntr in enumerate(n_sizes):
    for j, p in enumerate(ps):
        m = (n_train_all == ntr) & np.isclose(p_all, p)
        if not np.any(m):
            raise ValueError(f"Missing data for n_train={ntr}, p={p}")
        a[i, j] = float(mean_all[m][0])
        da[i, j] = float(stderr_all[m][0])

# Ensure strictly positive errors for fssa
nonzero = da[da > 0]
min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

# Fixed zeta = 0; expanded nu grid
zeta_fixed = 0.0
rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)

best = (np.inf, None, None)
for rho_c in rho_grid:
    for nu in nu_grid:
        scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
        S = fssa.quality(scaled.x, scaled.y, scaled.dy)
        if S < best[0]:
            best = (S, rho_c, nu)

S_star, rho_c_star, nu_star = best
print(f"fixed zeta=0: rho_c={rho_c_star:.4f}, nu={nu_star:.3f}, S={S_star:.4f}")

scaled = fssa.scaledata(L, rho, a, da, rho_c_star, nu_star, zeta_fixed)

fig, ax = plt.subplots(figsize=(8, 6))
for i, ntr in enumerate(n_sizes):
    ax.plot(scaled.x[i], scaled.y[i], marker=".", linestyle="", label=f"n_train={ntr}")
ax.set_title(f"pyfssa collapse (zeta=0, S={S_star:.3f})")
ax.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={rho_c_star:.4f}$, $\nu={nu_star:.3f}$")
ax.set_ylabel(r"$\mathrm{mean\ test\ acc}(p)-\mathrm{mean\ test\ acc}(p^*)$")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()




## pyfssa collapse + manual collapse (two separate figures with dashed mean curve)


In [ ]:
# pyfssa collapse + manual collapse (two separate figures with dashed mean curve)
import csv
import numpy as np
import matplotlib.pyplot as plt
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy", "stderr_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            mean_acc = float(r["mean_test_accuracy"])
            stderr = float(r["stderr_test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, mean_acc, stderr))

if not rows:
    raise ValueError("No valid rows found in summary CSV.")

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
mean_all = arr[:, 2]
stderr_all = arr[:, 3]

# Keep only n_train >= 10000 (expected for this summary)
mask = n_train_all >= 10000
n_train_all = n_train_all[mask]
p_all = p_all[mask]
mean_all = mean_all[mask]
stderr_all = stderr_all[mask]
if n_train_all.size == 0:
    raise ValueError("No rows remain after filtering to n_train_samples >= 10000.")

n_sizes = np.array(sorted(np.unique(n_train_all)), dtype=int)
if n_sizes.size < 2:
    raise ValueError(f"Need >=2 n_train values for collapse, found {n_sizes.tolist()}")

# Keep only p values that exist for all n_train sizes
p_common = None
for ntr in n_sizes:
    pset = set(np.unique(p_all[n_train_all == ntr]).tolist())
    p_common = pset if p_common is None else (p_common & pset)
ps = np.array(sorted(p_common), dtype=float)
if ps.size < 2:
    raise ValueError("Need >=2 shared p values across n_train sizes.")

# Build a(L, rho) and da(L, rho)
L = n_sizes.astype(float)  # system size = n_train
rho = ps
nL, nR = L.size, rho.size
a = np.zeros((nL, nR), dtype=float)
da = np.zeros_like(a)

for i, ntr in enumerate(n_sizes):
    for j, p in enumerate(ps):
        m = (n_train_all == ntr) & np.isclose(p_all, p)
        if not np.any(m):
            raise ValueError(f"Missing data for n_train={ntr}, p={p}")
        a[i, j] = float(mean_all[m][0])
        da[i, j] = float(stderr_all[m][0])

# Ensure strictly positive errors for fssa
nonzero = da[da > 0]
min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

# pyfssa: fixed zeta = 0; expanded nu grid
zeta_fixed = 0.0
rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)

best = (np.inf, None, None)
for rho_c in rho_grid:
    for nu in nu_grid:
        scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
        S = fssa.quality(scaled.x, scaled.y, scaled.dy)
        if S < best[0]:
            best = (S, rho_c, nu)

S_star, rho_c_star, nu_star = best
print(f"pyfssa fixed zeta=0: rho_c={rho_c_star:.4f}, nu={nu_star:.3f}, S={S_star:.4f}")

scaled = fssa.scaledata(L, rho, a, da, rho_c_star, nu_star, zeta_fixed)

# manual collapse: minimize SSE across curves (same data)

def manual_collapse_quality(pc, nu):
    curves = {}
    xs_all = []
    for i, ntr in enumerate(L):
        p_vals = rho
        s_vals = a[i]
        if pc < p_vals.min() or pc > p_vals.max():
            return np.inf
        s_pc = np.interp(pc, p_vals, s_vals)
        x_vals = (p_vals - pc) * (ntr ** (1.0 / nu))
        y_vals = s_vals - s_pc
        curves[ntr] = (x_vals, y_vals)
        xs_all.append(x_vals)
    xs_all = np.unique(np.concatenate(xs_all))
    total = 0.0
    count = 0
    for x0 in xs_all:
        ys = []
        for x_vals, y_vals in curves.values():
            if x0 < x_vals.min() or x0 > x_vals.max():
                continue
            ys.append(np.interp(x0, x_vals, y_vals))
        if len(ys) < 2:
            continue
        ybar = float(np.mean(ys))
        total += float(np.sum((np.array(ys) - ybar) ** 2))
        count += len(ys)
    return total / count if count > 0 else np.inf

pc_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)
manual_best = (np.inf, None, None)
for pc in pc_grid:
    for nu in nu_grid:
        q = manual_collapse_quality(pc, nu)
        if q < manual_best[0]:
            manual_best = (q, pc, nu)

Q_star, pc_star, nu_star_m = manual_best
print(f"manual collapse: pc={pc_star:.4f}, nu={nu_star_m:.3f}, Q={Q_star:.4e}")

# Helper: compute dashed mean scaling curve by binning all collapsed points

def binned_mean_curve(x_vals, y_vals, bins=60):
    x_vals = np.asarray(x_vals)
    y_vals = np.asarray(y_vals)
    xmin, xmax = float(np.min(x_vals)), float(np.max(x_vals))
    edges = np.linspace(xmin, xmax, bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    y_mean = np.full_like(centers, np.nan, dtype=float)
    for i in range(bins):
        m = (x_vals >= edges[i]) & (x_vals < edges[i + 1])
        if np.any(m):
            y_mean[i] = float(np.mean(y_vals[m]))
    valid = np.isfinite(y_mean)
    return centers[valid], y_mean[valid]

# Figure 1: pyfssa collapse
fig1, ax1 = plt.subplots(figsize=(8, 6))
all_x, all_y = [], []
for i, ntr in enumerate(n_sizes):
    ax1.plot(scaled.x[i], scaled.y[i], marker=".", linestyle="", label=f"n_train={ntr}")
    all_x.append(scaled.x[i])
    all_y.append(scaled.y[i])
all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)
mx, my = binned_mean_curve(all_x, all_y, bins=80)
ax1.plot(mx, my, "k--", linewidth=1.5, label="mean scaling")
ax1.set_title(f"pyfssa (zeta=0, S={S_star:.3f}) | width = 512, depth = 1")
ax1.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={rho_c_star:.4f}$, $\nu={nu_star:.3f}$")
ax1.set_ylabel(r"$\mathrm{mean\ test\ acc}(p)-\mathrm{mean\ test\ acc}(p^*)$")
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Figure 2: manual collapse
fig2, ax2 = plt.subplots(figsize=(8, 6))
all_x, all_y = [], []
for i, ntr in enumerate(n_sizes):
    p_vals = rho
    s_vals = a[i]
    s_pc = np.interp(pc_star, p_vals, s_vals)
    x_vals = (p_vals - pc_star) * (ntr ** (1.0 / nu_star_m))
    y_vals = s_vals - s_pc
    ax2.plot(x_vals, y_vals, marker=".", linestyle="", label=f"n_train={ntr}")
    all_x.append(x_vals)
    all_y.append(y_vals)
all_x = np.concatenate(all_x)
all_y = np.concatenate(all_y)
mx, my = binned_mean_curve(all_x, all_y, bins=80)
ax2.plot(mx, my, "k--", linewidth=1.5, label="mean scaling")
ax2.set_title(f"manual (Q={Q_star:.3e}) | width = 512, depth = 1")
ax2.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={pc_star:.4f}$, $\nu={nu_star_m:.3f}$")
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()




## pyfssa collapse + manual collapse (two separate figures)


In [ ]:
# pyfssa collapse + manual collapse (two separate figures)
import csv
import numpy as np
import matplotlib.pyplot as plt
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy", "stderr_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            mean_acc = float(r["mean_test_accuracy"])
            stderr = float(r["stderr_test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, mean_acc, stderr))

if not rows:
    raise ValueError("No valid rows found in summary CSV.")

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
mean_all = arr[:, 2]
stderr_all = arr[:, 3]

# Keep only n_train >= 10000 (expected for this summary)
mask = n_train_all >= 10000
n_train_all = n_train_all[mask]
p_all = p_all[mask]
mean_all = mean_all[mask]
stderr_all = stderr_all[mask]
if n_train_all.size == 0:
    raise ValueError("No rows remain after filtering to n_train_samples >= 10000.")

n_sizes = np.array(sorted(np.unique(n_train_all)), dtype=int)
if n_sizes.size < 2:
    raise ValueError(f"Need >=2 n_train values for collapse, found {n_sizes.tolist()}")

# Keep only p values that exist for all n_train sizes
p_common = None
for ntr in n_sizes:
    pset = set(np.unique(p_all[n_train_all == ntr]).tolist())
    p_common = pset if p_common is None else (p_common & pset)
ps = np.array(sorted(p_common), dtype=float)
if ps.size < 2:
    raise ValueError("Need >=2 shared p values across n_train sizes.")

# Build a(L, rho) and da(L, rho)
L = n_sizes.astype(float)  # system size = n_train
rho = ps
nL, nR = L.size, rho.size
a = np.zeros((nL, nR), dtype=float)
da = np.zeros_like(a)

for i, ntr in enumerate(n_sizes):
    for j, p in enumerate(ps):
        m = (n_train_all == ntr) & np.isclose(p_all, p)
        if not np.any(m):
            raise ValueError(f"Missing data for n_train={ntr}, p={p}")
        a[i, j] = float(mean_all[m][0])
        da[i, j] = float(stderr_all[m][0])

# Ensure strictly positive errors for fssa
nonzero = da[da > 0]
min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

# pyfssa: fixed zeta = 0; expanded nu grid
zeta_fixed = 0.0
rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)

best = (np.inf, None, None)
for rho_c in rho_grid:
    for nu in nu_grid:
        scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
        S = fssa.quality(scaled.x, scaled.y, scaled.dy)
        if S < best[0]:
            best = (S, rho_c, nu)

S_star, rho_c_star, nu_star = best
print(f"pyfssa fixed zeta=0: rho_c={rho_c_star:.4f}, nu={nu_star:.3f}, S={S_star:.4f}")

scaled = fssa.scaledata(L, rho, a, da, rho_c_star, nu_star, zeta_fixed)

# manual collapse: minimize SSE across curves (same data)

def manual_collapse_quality(pc, nu):
    curves = {}
    xs_all = []
    for i, ntr in enumerate(L):
        p_vals = rho
        s_vals = a[i]
        if pc < p_vals.min() or pc > p_vals.max():
            return np.inf
        s_pc = np.interp(pc, p_vals, s_vals)
        x_vals = (p_vals - pc) * (ntr ** (1.0 / nu))
        y_vals = s_vals - s_pc
        curves[ntr] = (x_vals, y_vals)
        xs_all.append(x_vals)
    xs_all = np.unique(np.concatenate(xs_all))
    total = 0.0
    count = 0
    for x0 in xs_all:
        ys = []
        for x_vals, y_vals in curves.values():
            if x0 < x_vals.min() or x0 > x_vals.max():
                continue
            ys.append(np.interp(x0, x_vals, y_vals))
        if len(ys) < 2:
            continue
        ybar = float(np.mean(ys))
        total += float(np.sum((np.array(ys) - ybar) ** 2))
        count += len(ys)
    return total / count if count > 0 else np.inf

pc_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)
manual_best = (np.inf, None, None)
for pc in pc_grid:
    for nu in nu_grid:
        q = manual_collapse_quality(pc, nu)
        if q < manual_best[0]:
            manual_best = (q, pc, nu)

Q_star, pc_star, nu_star_m = manual_best
print(f"manual collapse: pc={pc_star:.4f}, nu={nu_star_m:.3f}, Q={Q_star:.4e}")

# Figure 1: pyfssa collapse
fig1, ax1 = plt.subplots(figsize=(8, 6))
for i, ntr in enumerate(n_sizes):
    ax1.plot(scaled.x[i], scaled.y[i], marker=".", linestyle="", label=f"n_train={ntr}")
ax1.set_title(f"pyfssa (zeta=0, S={S_star:.3f})")
ax1.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={rho_c_star:.4f}$, $\nu={nu_star:.3f}$")
ax1.set_ylabel(r"$\mathrm{mean\ test\ acc}(p)-\mathrm{mean\ test\ acc}(p^*)$")
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Figure 2: manual collapse
fig2, ax2 = plt.subplots(figsize=(8, 6))
for i, ntr in enumerate(n_sizes):
    p_vals = rho
    s_vals = a[i]
    s_pc = np.interp(pc_star, p_vals, s_vals)
    x_vals = (p_vals - pc_star) * (ntr ** (1.0 / nu_star_m))
    y_vals = s_vals - s_pc
    ax2.plot(x_vals, y_vals, marker=".", linestyle="", label=f"n_train={ntr}")
ax2.set_title(f"manual (Q={Q_star:.3e})")
ax2.set_xlabel(rf"$x=(p-p^*)n_{{train}}^{{1/\nu}}$, $p^*={pc_star:.4f}$, $\nu={nu_star_m:.3f}$")
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)
plt.tight_layout()
plt.show()




## Robustness checks: tighten p-window and leave-one-size-out (sanity-check p* and nu stability)


In [ ]:
# Robustness checks: tighten p-window and leave-one-size-out (sanity-check p* and nu stability)
import csv
import numpy as np
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy", "stderr_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            mean_acc = float(r["mean_test_accuracy"])
            stderr = float(r["stderr_test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, mean_acc, stderr))

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
mean_all = arr[:, 2]
stderr_all = arr[:, 3]

# helper for pyfssa fit with fixed zeta

def pyfssa_fit(n_train_vals, p_vals, mean_vals, stderr_vals):
    n_sizes = np.array(sorted(np.unique(n_train_vals)), dtype=int)
    if n_sizes.size < 2:
        return None
    # common p across sizes
    p_common = None
    for ntr in n_sizes:
        pset = set(np.unique(p_vals[n_train_vals == ntr]).tolist())
        p_common = pset if p_common is None else (p_common & pset)
    ps = np.array(sorted(p_common), dtype=float)
    if ps.size < 2:
        return None

    # build a(L, rho), da(L, rho)
    L = n_sizes.astype(float)
    rho = ps
    a = np.zeros((L.size, rho.size), dtype=float)
    da = np.zeros_like(a)
    for i, ntr in enumerate(n_sizes):
        for j, p in enumerate(ps):
            m = (n_train_vals == ntr) & np.isclose(p_vals, p)
            if not np.any(m):
                return None
            a[i, j] = float(mean_vals[m][0])
            da[i, j] = float(stderr_vals[m][0])

    # ensure positive errors
    nonzero = da[da > 0]
    min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
    da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

    zeta_fixed = 0.0
    rho_grid = np.linspace(rho.min() + 1e-3, rho.max() - 1e-3, 60)
    nu_grid = np.linspace(0.2, 10.0, 60)
    best = (np.inf, None, None)
    for rho_c in rho_grid:
        for nu in nu_grid:
            scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
            S = fssa.quality(scaled.x, scaled.y, scaled.dy)
            if S < best[0]:
                best = (S, rho_c, nu)
    S_star, rho_c_star, nu_star = best
    return {
        "rho_c": float(rho_c_star),
        "nu": float(nu_star),
        "S": float(S_star),
        "n_train": n_sizes.tolist(),
        "p_min": float(rho.min()),
        "p_max": float(rho.max()),
    }

# 1) Tighten p-window around the knee
p_window = (0.92, 0.98)
mask = (p_all >= p_window[0]) & (p_all <= p_window[1])
res_window = pyfssa_fit(n_train_all[mask], p_all[mask], mean_all[mask], stderr_all[mask])
print("p-window", p_window, "->", res_window)

# 2) Leave-one-size-out fits
unique_sizes = sorted(np.unique(n_train_all).tolist())
for drop in unique_sizes:
    mask = n_train_all != drop
    res = pyfssa_fit(n_train_all[mask], p_all[mask], mean_all[mask], stderr_all[mask])
    print(f"drop n_train={drop} ->", res)




## Bootstrap uncertainty for pyfssa fit (p* and nu)


In [ ]:
# Bootstrap uncertainty for pyfssa fit (p* and nu)
# We resample p-points with replacement and refit to estimate parameter variability.
import csv
import numpy as np
import importlib

# --- compat patches (numpy + scipy) ---
if not hasattr(np, "int"):
    np.int = int

try:
    import scipy.optimize.optimize as _opt_mod
    _opt_mod = importlib.reload(_opt_mod)
    from scipy.optimize import _optimize as _opt

    for name in ("OptimizeResult", "_status_message", "wrap_function"):
        if not hasattr(_opt_mod, name) and hasattr(_opt, name):
            setattr(_opt_mod, name, getattr(_opt, name))

    if not hasattr(_opt_mod, "wrap_function"):
        def _wrap_function(function, args):
            ncalls = [0]
            def function_wrapper(x):
                ncalls[0] += 1
                return function(x, *args)
            return ncalls, function_wrapper
        _opt_mod.wrap_function = _wrap_function
except Exception as _exc:
    print("scipy monkeypatch failed:", _exc)

import fssa

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy", "stderr_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            ntr = int(float(r["n_train_samples"]))
            p = float(r["p"])
            mean_acc = float(r["mean_test_accuracy"])
            stderr = float(r["stderr_test_accuracy"])
        except Exception:
            continue
        rows.append((ntr, p, mean_acc, stderr))

arr = np.array(rows, dtype=float)
n_train_all = arr[:, 0].astype(int)
p_all = arr[:, 1]
mean_all = arr[:, 2]
stderr_all = arr[:, 3]

# Keep only n_train >= 10000
mask = n_train_all >= 10000
n_train_all = n_train_all[mask]
p_all = p_all[mask]
mean_all = mean_all[mask]
stderr_all = stderr_all[mask]

# Use only p values present for all n_train sizes
n_sizes = np.array(sorted(np.unique(n_train_all)), dtype=int)
if n_sizes.size < 2:
    raise ValueError("Need >=2 n_train values for bootstrap.")

p_common = None
for ntr in n_sizes:
    pset = set(np.unique(p_all[n_train_all == ntr]).tolist())
    p_common = pset if p_common is None else (p_common & pset)
ps = np.array(sorted(p_common), dtype=float)
if ps.size < 2:
    raise ValueError("Need >=2 shared p values across n_train sizes.")

# Build lookup dict: (n_train, p) -> (mean, stderr)
lookup = {(int(ntr), float(p)): (float(mu), float(se)) for ntr, p, mu, se in zip(n_train_all, p_all, mean_all, stderr_all)}

zeta_fixed = 0.0
rho_grid = np.linspace(ps.min() + 1e-3, ps.max() - 1e-3, 60)
nu_grid = np.linspace(0.2, 10.0, 60)

# Fit helper on a selected p-set

def fit_on_p_subset(p_subset):
    L = n_sizes.astype(float)
    rho = np.array(sorted(p_subset), dtype=float)
    a = np.zeros((L.size, rho.size), dtype=float)
    da = np.zeros_like(a)
    for i, ntr in enumerate(n_sizes):
        for j, p in enumerate(rho):
            mu, se = lookup[(int(ntr), float(p))]
            a[i, j] = mu
            da[i, j] = se
    nonzero = da[da > 0]
    min_nonzero = float(np.min(nonzero)) if nonzero.size else 1e-6
    da = np.clip(np.where(np.isfinite(da), da, min_nonzero), 1e-6, None)

    best = (np.inf, None, None)
    for rho_c in rho_grid:
        for nu in nu_grid:
            scaled = fssa.scaledata(L, rho, a, da, rho_c, nu, zeta_fixed)
            S = fssa.quality(scaled.x, scaled.y, scaled.dy)
            if S < best[0]:
                best = (S, rho_c, nu)
    S_star, rho_c_star, nu_star = best
    return float(rho_c_star), float(nu_star), float(S_star)

# Bootstrap by resampling p values with replacement
rng = np.random.default_rng(1234)
B = 200  # bootstrap samples
p_star = []
nu_star = []
S_star = []
for _ in range(B):
    p_subset = rng.choice(ps, size=len(ps), replace=True)
    # use unique p's to avoid degenerate duplicates
    p_subset = np.unique(p_subset)
    if p_subset.size < 2:
        continue
    rho_c, nu, S = fit_on_p_subset(p_subset)
    if np.isfinite(rho_c) and np.isfinite(nu) and np.isfinite(S):
        p_star.append(rho_c)
        nu_star.append(nu)
        S_star.append(S)

p_star = np.array(p_star)
nu_star = np.array(nu_star)
S_star = np.array(S_star)

print(f"bootstrap samples: {len(p_star)}")
print(f"p*: mean={p_star.mean():.4f}, std={p_star.std(ddof=0):.4f}")
print(f"nu: mean={nu_star.mean():.3f}, std={nu_star.std(ddof=0):.3f}")
print(f"S:  mean={S_star.mean():.3f}, std={S_star.std(ddof=0):.3f}")




## Mean test accuracy vs n_train for selected p-values (nearest available; exclude 0.95)


In [ ]:
# Mean test accuracy vs n_train for selected p-values (nearest available; exclude 0.95)
import csv
import numpy as np
import matplotlib.pyplot as plt

summary_path = "results/data/results_summary_mlpns_infimnist2000000_cache-infimnist_cache_2000000_r25_e20_ntr10000-500000_nte10000-500000_p0.90-1.00_w512_d1_loss-cross_entropy_ntrain_sweep_relu_only_infimnist.csv"

rows = []
with open(summary_path, newline="") as f:
    reader = csv.DictReader(f)
    required = {"p", "n_train_samples", "mean_test_accuracy"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for r in reader:
        try:
            p = float(r["p"])
            ntr = int(float(r["n_train_samples"]))
            mean_acc = float(r["mean_test_accuracy"])
        except Exception:
            continue
        rows.append((p, ntr, mean_acc))

if not rows:
    raise ValueError("No valid rows found in summary CSV.")

arr = np.array(rows, dtype=float)
p_all = arr[:, 0]
n_all = arr[:, 1].astype(int)
acc_all = arr[:, 2]

available_p = sorted(np.unique(p_all))

# requested p values (exclude 0.95 explicitly)
requested = [0.0, 0.5, 0.8, 0.9, 0.94, 0.96, 0.97, 0.98, 0.99, 1.00]
requested = [p for p in requested if abs(p - 0.95) > 1e-12]

# map to closest available p, drop duplicates
mapped = []
for p in requested:
    closest = min(available_p, key=lambda q: abs(q - p))
    if abs(closest - 0.95) <= 1e-12:
        continue
    mapped.append(closest)

mapped = sorted(set(mapped))
print("available p:", available_p)
print("requested:", requested)
print("mapped to:", mapped)

# plot mean test accuracy vs n_train for each mapped p
plt.figure(figsize=(8, 5))
for p in mapped:
    m = np.isclose(p_all, p)
    ntr = n_all[m]
    acc = acc_all[m]
    # sort by n_train
    order = np.argsort(ntr)
    ntr = ntr[order]
    acc = acc[order]
    plt.plot(ntr, acc, marker='o', label=f"p={p:.2f}")

plt.xscale("log")
plt.xlabel("n_train")
plt.ylabel("mean test accuracy")
plt.title("Mean test accuracy vs n_train (nearest p values)")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()




## Interactive InfiMNIST alpha exploration (no saving)


In [ ]:
# Interactive InfiMNIST alpha exploration (no saving)
# `deform_k` changes the deformation while keeping the same base digit index.
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from src.nnet_models import _import_infimnist_module, _validate_infimnist_data_dir

# InfiMNIST uses MNIST sizes internally (train=60000, test=10000).
TRAINNUM = 60000
TESTNUM = 10000

infimnist = _import_infimnist_module()
_validate_infimnist_data_dir(infimnist)

alpha_slider = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=3.0,
    step=0.05,
    description='alpha',
    continuous_update=False,
)
idx_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=TRAINNUM - 1,
    step=1,
    description='digit_idx',
    continuous_update=False,
)
deform_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=200,
    step=1,
    description='deform_k',
    continuous_update=False,
)
translate_toggle = widgets.Checkbox(
    value=True,
    description='translate',
)

out = widgets.Output()

def _render(alpha, digit_idx, deform_k, translate):
    # Use an index >= TESTNUM+TRAINNUM so deformation path is taken.
    # Adding TRAINNUM * deform_k changes the deformation but keeps the same base digit.
    idx = TESTNUM + int(digit_idx) + int(deform_k) * TRAINNUM

    gen0 = infimnist.InfimnistGenerator(alpha=0.0, translate=bool(translate))
    gen1 = infimnist.InfimnistGenerator(alpha=float(alpha), translate=bool(translate))

    base_img, base_lbl = gen0.gen(np.asarray([idx], dtype=np.int64))
    img, lbl = gen1.gen(np.asarray([idx], dtype=np.int64))

    base_img = base_img.reshape(28, 28)
    img = img.reshape(28, 28)

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(6, 3))
        axes[0].imshow(base_img, cmap='gray')
        axes[0].set_title(f'alpha=0, label={int(base_lbl[0])}')
        axes[0].axis('off')
        axes[1].imshow(img, cmap='gray')
        axes[1].set_title(f'alpha={alpha:.2f}, label={int(lbl[0])}')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

ui = widgets.VBox([
    widgets.HBox([alpha_slider, translate_toggle]),
    idx_slider,
    deform_slider,
])

def _on_change(_):
    _render(alpha_slider.value, idx_slider.value, deform_slider.value, translate_toggle.value)

alpha_slider.observe(_on_change, names='value')
idx_slider.observe(_on_change, names='value')
deform_slider.observe(_on_change, names='value')
translate_toggle.observe(_on_change, names='value')

# Initial render
_render(alpha_slider.value, idx_slider.value, deform_slider.value, translate_toggle.value)

display(ui, out)


## Mini ntrain sweep (infimnist)


In [ ]:
# Mini ntrain sweep (infimnist)
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm import tqdm

from src.nnet_models import TrainConfig, train_and_evaluate

# Settings
activations = ["relu"]
model_types = ["mlp"]
ps = np.linspace(0.9, 1.0, 11)
width = 64
mlp_depth = 1
n_train_values = [10000, 50000, 100000, 500000, 1000000]
repeats = 20
epochs = 20
batch_size = 128
learning_rate = 1e-3
weight_decay = 0.0
loss_type = "cross_entropy"
corruption_mode = "replacement"

# Dataset setup (infimnist)
dataset_name = "infimnist"
dataset_dim = 2e7
infimnist_cache_dir = "data/infimnist_cache_2e7"
exact_sample_counts = True

results = []
start = datetime.now()

run_grid = [(n_train, p, r) for n_train in n_train_values for p in ps for r in range(repeats)]

for n_train, p, r in tqdm(run_grid, desc="mini ntrain sweep", unit="run"):
    cfg = TrainConfig(
        model_type="mlp",
        activation="relu",
        mlp_hidden_sizes=[width] * mlp_depth,
        p=float(p),
        sigma=0.0,
        corruption_mode=corruption_mode,
        loss_type=loss_type,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        num_workers=0,
        cpu_threads=1,
        brightness_scale=1.0,
        split_seed=1234 + r,
        seed = 1234 + r,
        dataset_name=dataset_name,
        dataset_dim=dataset_dim,
        infimnist_cache_dir=infimnist_cache_dir,
        exact_sample_counts=exact_sample_counts,
        exact_train_samples=n_train,
        exact_test_samples=n_train,
        max_train_samples=None,
        use_cuda=False,
    )
    res = train_and_evaluate(cfg)
    res["n_train_samples"] = n_train
    res["repeat"] = r
    results.append(res)

print(f"Completed mini sweep in {datetime.now() - start}")


## Depth sweep: reproduce run_experiments_noisy_training_data_depth_sweep.py outputs


In [ ]:
# Depth sweep: reproduce run_experiments_noisy_training_data_depth_sweep.py outputs
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_mlpds_r20_e50_tf0.500_p0.00-1.00_w64_d1-16_loss-cross_entropy_depth_sweep.csv")

if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# Summarize per (activation, model_type, corruption_mode, p, sigma, depth)
summary = {}
for r in rows:
    depth = len(eval(r["mlp_hidden_sizes"])) if r.get("mlp_hidden_sizes") else 0
    key = (
        r["activation"],
        r["model_type"],
        r["corruption_mode"],
        float(r["p"]),
        float(r["sigma"]),
        depth,
    )
    summary.setdefault(key, []).append(r)

summary_rows = []
for (act, model_type, corruption_mode, p, sigma, depth), items in summary.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)
    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_loss = float(np.mean(test_loss))
    std_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary_rows.append({
        "activation": act,
        "model_type": model_type,
        "corruption_mode": corruption_mode,
        "p": p,
        "sigma": sigma,
        "mlp_depth": depth,
        "repeats": n,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "stderr_test_accuracy": std_acc / math.sqrt(n) if n > 0 else 0.0,
        "mean_test_loss": mean_loss,
        "std_test_loss": std_loss,
        "stderr_test_loss": std_loss / math.sqrt(n) if n > 0 else 0.0,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
        "stderr_train_loss": std_train_loss / math.sqrt(n) if n > 0 else 0.0,
    })

# Plot (matches script style; no NTK plots)
model_types = sorted({r["model_type"] for r in summary_rows})
for model_type in model_types:
    sub = [r for r in summary_rows if r["model_type"] == model_type]
    activations = sorted({r["activation"] for r in sub})
    depths = sorted({int(r["mlp_depth"]) for r in sub})
    sweep_label = "p" if all(r["corruption_mode"] == "replacement" for r in sub) else "sigma"

    # mean test accuracy
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        for depth in depths:
            d = sorted(
                [r for r in sub if r["activation"] == act and int(r["mlp_depth"]) == depth],
                key=lambda r: float(r[sweep_label]),
            )
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_test_accuracy"]) for r in d],
                yerr=[float(r["stderr_test_accuracy"]) for r in d],
                marker="o",
                label=f"d={depth}",
            )
        ax.set_title(f"{model_type} / {act}")
        ax.set_xlabel(sweep_label)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("mean test accuracy")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # std test accuracy
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        for depth in depths:
            d = sorted(
                [r for r in sub if r["activation"] == act and int(r["mlp_depth"]) == depth],
                key=lambda r: float(r[sweep_label]),
            )
            ax.plot(
                [float(r[sweep_label]) for r in d],
                [float(r["std_test_accuracy"]) for r in d],
                marker="o",
                label=f"d={depth}",
            )
        ax.set_title(f"{model_type} / {act}")
        ax.set_xlabel(sweep_label)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("std test accuracy")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # mean test loss
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        for depth in depths:
            d = sorted(
                [r for r in sub if r["activation"] == act and int(r["mlp_depth"]) == depth],
                key=lambda r: float(r[sweep_label]),
            )
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_test_loss"]) for r in d],
                yerr=[float(r["stderr_test_loss"]) for r in d],
                marker="o",
                label=f"d={depth}",
            )
        ax.set_title(f"{model_type} / {act}")
        ax.set_xlabel(sweep_label)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("mean test loss")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # mean train loss
    fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 3), sharey=True)
    if len(activations) == 1:
        axes = [axes]
    for ax, act in zip(axes, activations):
        for depth in depths:
            d = sorted(
                [r for r in sub if r["activation"] == act and int(r["mlp_depth"]) == depth],
                key=lambda r: float(r[sweep_label]),
            )
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_train_loss"]) for r in d],
                yerr=[float(r["stderr_train_loss"]) for r in d],
                marker="o",
                label=f"d={depth}",
            )
        ax.set_title(f"{model_type} / {act}")
        ax.set_xlabel(sweep_label)
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel("mean train loss")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## Width sweep (relu): 2x2 summary grid


In [ ]:
# Width sweep (relu): 2x2 summary grid
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv")
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# Group by (width, p)
by_wp = {}
for r in rows:
    # width
    if r.get("mlp_width"):
        w = int(float(r["mlp_width"]))
    else:
        # fallback to first hidden size
        w = int(float(r["mlp_hidden_sizes"].strip("[]").split(",")[0]))
    p = float(r["p"])
    by_wp.setdefault((w, p), []).append(r)

summary = []
for (w, p), items in by_wp.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)

    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_test_loss = float(np.mean(test_loss))
    std_test_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary.append({
        "mlp_width": w,
        "p": p,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
    })

widths = sorted({r["mlp_width"] for r in summary})

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)

# (0,0) mean test acc
ax = axes[0, 0]
for w in widths:
    d = sorted([r for r in summary if r["mlp_width"] == w], key=lambda r: r["p"])
    ax.plot([r["p"] for r in d], [r["mean_test_accuracy"] for r in d], marker='.', label=f"w={w}")
ax.set_xlabel("p")
ax.set_ylabel("mean test accuracy")
ax.set_title("Mean test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (1,0) mean test std
ax = axes[1, 0]
for w in widths:
    d = sorted([r for r in summary if r["mlp_width"] == w], key=lambda r: r["p"])
    ax.plot([r["p"] for r in d], [r["std_test_accuracy"] for r in d], marker='.', label=f"w={w}")
ax.set_xlabel("p")
ax.set_ylabel("std test accuracy")
ax.set_title("Std test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (0,1) mean test loss
ax = axes[0, 1]
for w in widths:
    d = sorted([r for r in summary if r["mlp_width"] == w], key=lambda r: r["p"])
    ax.plot([r["p"] for r in d], [r["mean_test_loss"] for r in d], marker='.', label=f"w={w}")
ax.set_xlabel("p")
ax.set_ylabel("mean test loss")
ax.set_title("Mean test loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (1,1) mean train loss
ax = axes[1, 1]
for w in widths:
    d = sorted([r for r in summary if r["mlp_width"] == w], key=lambda r: r["p"])
    ax.plot([r["p"] for r in d], [r["mean_train_loss"] for r in d], marker='.', label=f"w={w}")
ax.set_xlabel("p")
ax.set_ylabel("mean train loss")
ax.set_title("Mean train loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Width sweep (full): 2x2 grids for relu and tanh


In [ ]:
# Width sweep (full): 2x2 grids for relu and tanh
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_width_sweep.csv")
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# Group by (activation, width, p)
by_awp = {}
for r in rows:
    act = r.get("activation", "")
    if r.get("mlp_width"):
        w = int(float(r["mlp_width"]))
    else:
        w = int(float(r["mlp_hidden_sizes"].strip("[]").split(",")[0]))
    p = float(r["p"])
    by_awp.setdefault((act, w, p), []).append(r)

summary = []
for (act, w, p), items in by_awp.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)

    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_test_loss = float(np.mean(test_loss))
    std_test_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary.append({
        "activation": act,
        "mlp_width": w,
        "p": p,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
    })

for act in ["relu", "tanh"]:
    sub = [r for r in summary if r["activation"] == act]
    if not sub:
        print(f"No rows found for activation={act}")
        continue
    widths = sorted({r["mlp_width"] for r in sub})

    fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)

    # (0,0) mean test acc
    ax = axes[0, 0]
    for w in widths:
        d = sorted([r for r in sub if r["mlp_width"] == w], key=lambda r: r["p"])
        ax.plot([r["p"] for r in d], [r["mean_test_accuracy"] for r in d], marker='.', label=f"w={w}")
    ax.set_xlabel("p")
    ax.set_ylabel("mean test accuracy")
    ax.set_title(f"{act}: Mean test accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    # (1,0) mean test std
    ax = axes[1, 0]
    for w in widths:
        d = sorted([r for r in sub if r["mlp_width"] == w], key=lambda r: r["p"])
        ax.plot([r["p"] for r in d], [r["std_test_accuracy"] for r in d], marker='.', label=f"w={w}")
    ax.set_xlabel("p")
    ax.set_ylabel("std test accuracy")
    ax.set_title(f"{act}: Std test accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    # (0,1) mean test loss
    ax = axes[0, 1]
    for w in widths:
        d = sorted([r for r in sub if r["mlp_width"] == w], key=lambda r: r["p"])
        ax.plot([r["p"] for r in d], [r["mean_test_loss"] for r in d], marker='.', label=f"w={w}")
    ax.set_xlabel("p")
    ax.set_ylabel("mean test loss")
    ax.set_title(f"{act}: Mean test loss")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    # (1,1) mean train loss
    ax = axes[1, 1]
    for w in widths:
        d = sorted([r for r in sub if r["mlp_width"] == w], key=lambda r: r["p"])
        ax.plot([r["p"] for r in d], [r["mean_train_loss"] for r in d], marker='.', label=f"w={w}")
    ax.set_xlabel("p")
    ax.set_ylabel("mean train loss")
    ax.set_title(f"{act}: Mean train loss")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


## Clean fraction noisy vs benchmark: reproduce grid plots from summary CSV


In [ ]:
# Clean fraction noisy vs benchmark: reproduce grid plots from summary CSV
import csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

summary_path = Path("results/data/results_summary_mlp_rn100_rc100_rfc1_e20_tf0.500_clean0.000-0.500_p0.00-1.00_w64_d1_loss-cross_entropy_clean_frac_noisy_vs_benchmark.csv")
if not summary_path.exists():
    raise FileNotFoundError(summary_path)

rows = list(csv.DictReader(open(summary_path, newline="")))

# Split modes
noisy = [r for r in rows if r.get("mode") == "noisy_complement"]
bench = [r for r in rows if r.get("mode") == "clean_only"]
full_clean = [r for r in rows if r.get("mode") == "full_clean_once"]

if not noisy:
    raise ValueError("No noisy_complement rows found in summary CSV.")

# Locked colors (match script)
color_noisy = "#1f77b4"
color_clean_subset = "#000000"
color_full_clean = "#666666"

activations = sorted({r["activation"] for r in noisy})
clean_fracs = sorted({float(r["clean_frac"]) for r in noisy if float(r["clean_frac"]) > 0})
if not clean_fracs:
    clean_fracs = sorted({float(r["clean_frac"]) for r in noisy})

# lookup tables
bench_lookup = {}
for r in bench:
    bench_lookup[(r["activation"], float(r["clean_frac"]))] = float(r["mean_test_accuracy"])

full_clean_lookup = {}
for r in full_clean:
    full_clean_lookup[r["activation"]] = float(r["mean_test_accuracy"])

nrows = len(clean_fracs)
ncols = len(activations)
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), sharex=True, sharey=True)
if nrows == 1:
    axes = [axes]

for r, frac in enumerate(clean_fracs):
    for c, act in enumerate(activations):
        ax = axes[r][c] if ncols > 1 else axes[r]
        d = sorted(
            [x for x in noisy if x["activation"] == act and float(x["clean_frac"]) == frac],
            key=lambda x: float(x["p"]),
        )
        if not d:
            continue

        ax.errorbar(
            [float(x["p"]) for x in d],
            [float(x["mean_test_accuracy"]) for x in d],
            yerr=[float(x["stderr_test_accuracy"]) for x in d],
            marker='.',
            color=color_noisy,
            label='clean training subset + noisy complementary set',
        )

        bench_y = bench_lookup.get((act, frac))
        if bench_y is not None:
            ax.axhline(
                bench_y,
                color=color_clean_subset,
                linestyle='--',
                linewidth=1.5,
                label='clean training subset benchmark',
            )

        full_clean_y = full_clean_lookup.get(act)
        if full_clean_y is not None:
            ax.axhline(
                full_clean_y,
                color=color_full_clean,
                linestyle=':',
                linewidth=1.8,
                label='entired clean dataset',
            )

        ax.set_title(f"{act} / frac={frac:.3f}")
        ax.set_xlabel('p')
        if c == 0:
            ax.set_ylabel('mean test accuracy')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

fig.suptitle('Clean subset + noisy complementary set vs clean-only benchmark', y=1.02)
plt.tight_layout()
plt.show()


## Width sweep (relu-only): reproduce run_experiments_noisy_training_data_width_sweep.py plots


In [ ]:
# Width sweep (relu-only): reproduce run_experiments_noisy_training_data_width_sweep.py plots
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_mlpws_r100_e20_tf0.500_p0.90-1.00_w128-1024_d1_loss-cross_entropy_width_sweep_relu_only.csv")
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# summarize per (activation, width, p)
summary = {}
for r in rows:
    act = r.get("activation", "")
    if r.get("mlp_width"):
        w = int(float(r["mlp_width"]))
    else:
        w = int(float(r["mlp_hidden_sizes"].strip("[]").split(",")[0]))
    p = float(r["p"])
    key = (act, w, p)
    summary.setdefault(key, []).append(r)

summary_rows = []
for (act, w, p), items in summary.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)

    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_test_loss = float(np.mean(test_loss))
    std_test_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary_rows.append({
        "activation": act,
        "mlp_width": w,
        "p": p,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "stderr_test_accuracy": std_acc / math.sqrt(n) if n > 0 else 0.0,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "stderr_test_loss": std_test_loss / math.sqrt(n) if n > 0 else 0.0,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
        "stderr_train_loss": std_train_loss / math.sqrt(n) if n > 0 else 0.0,
    })

summary_rows.sort(key=lambda r: (r["activation"], r["mlp_width"], r["p"]))

# Plot (mean acc, std acc, mean test loss, mean train loss) + FSS collapse
activations = sorted({r["activation"] for r in summary_rows})
widths = sorted({int(r["mlp_width"]) for r in summary_rows})

# Mean test accuracy
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_accuracy"]) for r in d],
        yerr=[float(r["stderr_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("mean test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Std test accuracy
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.plot(
        [float(r["p"]) for r in d],
        [float(r["std_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Std test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("std test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Mean test loss
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_loss"]) for r in d],
        yerr=[float(r["stderr_test_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test loss")
ax.set_xlabel("p")
ax.set_ylabel("mean test loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Mean train loss
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_train_loss"]) for r in d],
        yerr=[float(r["stderr_train_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean train loss")
ax.set_xlabel("p")
ax.set_ylabel("mean train loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# FSS collapse (width as system size)
# Build data by width
width_data = {}
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    p_vals = np.array([float(r["p"]) for r in d], dtype=float)
    s_vals = np.array([float(r["mean_test_accuracy"]) for r in d], dtype=float)
    if len(p_vals) >= 3:
        width_data[w] = (p_vals, s_vals)

if len(width_data) >= 2:
    p_min = min(v[0].min() for v in width_data.values())
    p_max = max(v[0].max() for v in width_data.values())
    pc_grid = np.linspace(p_min, p_max, 41)
    nu_grid = np.linspace(0.2, 5.0, 41)

    best = (float("inf"), None, None)
    for pc in pc_grid:
        for nu in nu_grid:
            curves = {}
            xs_all = []
            for width, (p_vals, s_vals) in width_data.items():
                if pc < p_vals.min() or pc > p_vals.max():
                    continue
                s_pc = np.interp(pc, p_vals, s_vals)
                x_vals = (p_vals - pc) * (width ** (1.0 / nu))
                y_vals = s_vals - s_pc
                curves[width] = (x_vals, y_vals)
                xs_all.append(x_vals)
            if not curves:
                continue
            xs_all = np.unique(np.concatenate(xs_all))
            total = 0.0
            count = 0
            for x in xs_all:
                ys = []
                for x_vals, y_vals in curves.values():
                    if x < x_vals.min() or x > x_vals.max():
                        continue
                    ys.append(np.interp(x, x_vals, y_vals))
                if len(ys) < 2:
                    continue
                ybar = float(np.mean(ys))
                total += float(np.sum((np.array(ys) - ybar) ** 2))
                count += len(ys)
            if count == 0:
                continue
            q = total / count
            if q < best[0]:
                best = (q, pc, nu)

    _, pc_opt, nu_opt = best
    if pc_opt is not None and nu_opt is not None:
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for width, (p_vals, s_vals) in width_data.items():
            s_pc = np.interp(pc_opt, p_vals, s_vals)
            x_vals = (p_vals - pc_opt) * (width ** (1.0 / nu_opt))
            y_vals = s_vals - s_pc
            ax.plot(x_vals, y_vals, marker='o', label=f"w={width}")
        ax.set_title(f"FSS collapse: pc={pc_opt:.3f}, nu={nu_opt:.2f}")
        ax.set_xlabel(r"(p - p*) w^{1/nu}")
        ax.set_ylabel(r"mean acc(p) - mean acc(p*)")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()


## Width sweep (relu-only, rmax100): reproduce width-sweep plots


In [ ]:
# Width sweep (relu-only, rmax100): reproduce width-sweep plots
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_mlpws_rmax100_e20_tf0.500_p0.00-1.00_w64-64_d1_loss-cross_entropy_width_sweep_relu_only.csv")
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# summarize per (activation, width, p)
summary = {}
for r in rows:
    act = r.get("activation", "")
    if r.get("mlp_width"):
        w = int(float(r["mlp_width"]))
    else:
        w = int(float(r["mlp_hidden_sizes"].strip("[]").split(",")[0]))
    p = float(r["p"])
    key = (act, w, p)
    summary.setdefault(key, []).append(r)

summary_rows = []
for (act, w, p), items in summary.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)

    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_test_loss = float(np.mean(test_loss))
    std_test_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary_rows.append({
        "activation": act,
        "mlp_width": w,
        "p": p,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "stderr_test_accuracy": std_acc / math.sqrt(n) if n > 0 else 0.0,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "stderr_test_loss": std_test_loss / math.sqrt(n) if n > 0 else 0.0,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
        "stderr_train_loss": std_train_loss / math.sqrt(n) if n > 0 else 0.0,
    })

summary_rows.sort(key=lambda r: (r["activation"], r["mlp_width"], r["p"]))

widths = sorted({int(r["mlp_width"]) for r in summary_rows})

# Mean test accuracy
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_accuracy"]) for r in d],
        yerr=[float(r["stderr_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("mean test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Std test accuracy
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.plot(
        [float(r["p"]) for r in d],
        [float(r["std_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Std test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("std test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Mean test loss
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_loss"]) for r in d],
        yerr=[float(r["stderr_test_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test loss")
ax.set_xlabel("p")
ax.set_ylabel("mean test loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Mean train loss
fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_train_loss"]) for r in d],
        yerr=[float(r["stderr_train_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean train loss")
ax.set_xlabel("p")
ax.set_ylabel("mean train loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# FSS collapse (width as system size)
width_data = {}
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    p_vals = np.array([float(r["p"]) for r in d], dtype=float)
    s_vals = np.array([float(r["mean_test_accuracy"]) for r in d], dtype=float)
    if len(p_vals) >= 3:
        width_data[w] = (p_vals, s_vals)

if len(width_data) >= 2:
    p_min = min(v[0].min() for v in width_data.values())
    p_max = max(v[0].max() for v in width_data.values())
    pc_grid = np.linspace(p_min, p_max, 41)
    nu_grid = np.linspace(0.2, 5.0, 41)

    best = (float("inf"), None, None)
    for pc in pc_grid:
        for nu in nu_grid:
            curves = {}
            xs_all = []
            for width, (p_vals, s_vals) in width_data.items():
                if pc < p_vals.min() or pc > p_vals.max():
                    continue
                s_pc = np.interp(pc, p_vals, s_vals)
                x_vals = (p_vals - pc) * (width ** (1.0 / nu))
                y_vals = s_vals - s_pc
                curves[width] = (x_vals, y_vals)
                xs_all.append(x_vals)
            if not curves:
                continue
            xs_all = np.unique(np.concatenate(xs_all))
            total = 0.0
            count = 0
            for x in xs_all:
                ys = []
                for x_vals, y_vals in curves.values():
                    if x < x_vals.min() or x > x_vals.max():
                        continue
                    ys.append(np.interp(x, x_vals, y_vals))
                if len(ys) < 2:
                    continue
                ybar = float(np.mean(ys))
                total += float(np.sum((np.array(ys) - ybar) ** 2))
                count += len(ys)
            if count == 0:
                continue
            q = total / count
            if q < best[0]:
                best = (q, pc, nu)

    _, pc_opt, nu_opt = best
    if pc_opt is not None and nu_opt is not None:
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for width, (p_vals, s_vals) in width_data.items():
            s_pc = np.interp(pc_opt, p_vals, s_vals)
            x_vals = (p_vals - pc_opt) * (width ** (1.0 / nu_opt))
            y_vals = s_vals - s_pc
            ax.plot(x_vals, y_vals, marker='o', label=f"w={width}")
        ax.set_title(f"FSS collapse: pc={pc_opt:.3f}, nu={nu_opt:.2f}")
        ax.set_xlabel(r"(p - p*) w^{1/nu}")
        ax.set_ylabel(r"mean acc(p) - mean acc(p*)")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()


## Width sweep (relu-only, rmax100): 2x2 grid plots


In [ ]:
# Width sweep (relu-only, rmax100): 2x2 grid plots
import csv
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

per_run_path = Path("results/data/results_per_run_mlpws_rmax100_e20_tf0.500_p0.00-1.00_w64-64_d1_loss-cross_entropy_width_sweep_relu_only.csv")
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))

# summarize per (activation, width, p)
summary = {}
for r in rows:
    act = r.get("activation", "")
    if r.get("mlp_width"):
        w = int(float(r["mlp_width"]))
    else:
        w = int(float(r["mlp_hidden_sizes"].strip("[]").split(",")[0]))
    p = float(r["p"])
    key = (act, w, p)
    summary.setdefault(key, []).append(r)

summary_rows = []
for (act, w, p), items in summary.items():
    test_acc = [float(x["test_accuracy"]) for x in items]
    test_loss = [float(x["test_loss"]) for x in items]
    train_loss = [float(x.get("train_loss", 0.0)) for x in items]
    n = len(items)

    mean_acc = float(np.mean(test_acc))
    std_acc = float(np.std(test_acc, ddof=1)) if n > 1 else 0.0
    mean_test_loss = float(np.mean(test_loss))
    std_test_loss = float(np.std(test_loss, ddof=1)) if n > 1 else 0.0
    mean_train_loss = float(np.mean(train_loss))
    std_train_loss = float(np.std(train_loss, ddof=1)) if n > 1 else 0.0

    summary_rows.append({
        "activation": act,
        "mlp_width": w,
        "p": p,
        "mean_test_accuracy": mean_acc,
        "std_test_accuracy": std_acc,
        "stderr_test_accuracy": std_acc / math.sqrt(n) if n > 0 else 0.0,
        "mean_test_loss": mean_test_loss,
        "std_test_loss": std_test_loss,
        "stderr_test_loss": std_test_loss / math.sqrt(n) if n > 0 else 0.0,
        "mean_train_loss": mean_train_loss,
        "std_train_loss": std_train_loss,
        "stderr_train_loss": std_train_loss / math.sqrt(n) if n > 0 else 0.0,
    })

summary_rows.sort(key=lambda r: (r["activation"], r["mlp_width"], r["p"]))

widths = sorted({int(r["mlp_width"]) for r in summary_rows})

fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=False, sharey=False)

# (0,0) mean test accuracy
ax = axes[0, 0]
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_accuracy"]) for r in d],
        yerr=[float(r["stderr_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("mean test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (1,0) std test accuracy
ax = axes[1, 0]
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.plot(
        [float(r["p"]) for r in d],
        [float(r["std_test_accuracy"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Std test accuracy")
ax.set_xlabel("p")
ax.set_ylabel("std test accuracy")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (0,1) mean test loss
ax = axes[0, 1]
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_test_loss"]) for r in d],
        yerr=[float(r["stderr_test_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean test loss")
ax.set_xlabel("p")
ax.set_ylabel("mean test loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

# (1,1) mean train loss
ax = axes[1, 1]
for w in widths:
    d = [r for r in summary_rows if int(r["mlp_width"]) == w]
    d = sorted(d, key=lambda r: float(r["p"]))
    ax.errorbar(
        [float(r["p"]) for r in d],
        [float(r["mean_train_loss"]) for r in d],
        yerr=[float(r["stderr_train_loss"]) for r in d],
        marker="o",
        label=f"w={w}",
    )
ax.set_title("Mean train loss")
ax.set_xlabel("p")
ax.set_ylabel("mean train loss")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Batch-size sweep: reproduce script-style characterization plots from per-run CSV


In [ ]:
# Batch-size sweep: reproduce script-style characterization plots from per-run CSV
import csv
import math
import re
import statistics
from pathlib import Path

import matplotlib.pyplot as plt

per_run_path = Path(
    "results/data/results_per_run_mlpbs_rmax100_e20_tf0.500_p0.90-1.00_b32-256_w128_d1_loss-cross_entropy_batchsize_sweep_relu_only.csv"
)
if not per_run_path.exists():
    raise FileNotFoundError(per_run_path)

rows = list(csv.DictReader(open(per_run_path, newline="")))
if not rows:
    raise ValueError("Per-run CSV is empty.")

fieldnames = set(rows[0].keys())
has_batch_size = ("batch_size" in fieldnames) or ("batch" in fieldnames)

# Older CSVs (written before bug fix) can be missing batch size per row.
# If so, we can only recover when the filename encodes a single batch size (bX-X).
batch_fallback = None
if not has_batch_size:
    m = re.search(r"_b(\d+)-(\d+)_", per_run_path.name)
    if m and m.group(1) == m.group(2):
        batch_fallback = int(m.group(1))
    else:
        raise ValueError(
            "This per-run CSV has no 'batch_size' column and encodes multiple batch sizes in the filename. "
            "Per-row batch membership is unrecoverable. Re-run the experiment with the fixed script to regenerate CSV."
        )

# Group and summarize exactly like the experiment script does before plotting.
grouped = {}
for row in rows:
    if has_batch_size:
        batch_size = int(float(row.get("batch_size", row.get("batch"))))
    else:
        batch_size = batch_fallback

    key = (
        row["activation"],
        row["model_type"],
        row["corruption_mode"],
        float(row["p"]),
        float(row["sigma"]),
        batch_size,
    )
    grouped.setdefault(key, []).append(row)

summary_rows = []
for (activation, model_type, corruption_mode, p, sigma, batch_size), items in grouped.items():
    accs = [float(r["test_accuracy"]) for r in items]
    test_losses = [float(r["test_loss"]) for r in items]
    train_losses = [float(r.get("train_loss", 0.0)) for r in items]

    std_acc = statistics.pstdev(accs) if len(accs) > 1 else 0.0
    std_test_loss = statistics.pstdev(test_losses) if len(test_losses) > 1 else 0.0
    std_train_loss = statistics.pstdev(train_losses) if len(train_losses) > 1 else 0.0
    denom = math.sqrt(len(items)) if len(items) > 1 else None

    summary_rows.append({
        "activation": activation,
        "model_type": model_type,
        "corruption_mode": corruption_mode,
        "p": p,
        "sigma": sigma,
        "batch_size": batch_size,
        "repeats": len(items),
        "mean_test_accuracy": statistics.mean(accs),
        "std_test_accuracy": std_acc,
        "stderr_test_accuracy": (std_acc / denom) if denom else 0.0,
        "mean_test_loss": statistics.mean(test_losses),
        "std_test_loss": std_test_loss,
        "stderr_test_loss": (std_test_loss / denom) if denom else 0.0,
        "mean_train_loss": statistics.mean(train_losses),
        "std_train_loss": std_train_loss,
        "stderr_train_loss": (std_train_loss / denom) if denom else 0.0,
    })

summary_rows.sort(
    key=lambda r: (
        r["model_type"],
        r["activation"],
        r["corruption_mode"],
        r["batch_size"],
        r["p"],
        r["sigma"],
    )
)

model_types = sorted({r["model_type"] for r in summary_rows})

for model_type in model_types:
    sub = [r for r in summary_rows if r["model_type"] == model_type]
    activations = sorted({r["activation"] for r in sub})
    corruption_modes = {r["corruption_mode"] for r in sub}
    sweep_label = "sigma" if (len(corruption_modes) == 1 and "additive" in corruption_modes) else "p"

    for act in activations:
        act_rows = [r for r in sub if r["activation"] == act]
        batch_sizes = sorted({int(r["batch_size"]) for r in act_rows})

        # 1) mean test accuracy
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for b in batch_sizes:
            d = sorted([r for r in act_rows if int(r["batch_size"]) == b], key=lambda r: float(r[sweep_label]))
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_test_accuracy"]) for r in d],
                yerr=[float(r["stderr_test_accuracy"]) for r in d],
                marker=".",
                label=f"b={b}",
            )
        ax.set_title(f"{model_type} / {act} -- mean test accuracy")
        ax.set_xlabel(sweep_label)
        ax.set_ylabel("mean test accuracy")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        # 2) std test accuracy
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for b in batch_sizes:
            d = sorted([r for r in act_rows if int(r["batch_size"]) == b], key=lambda r: float(r[sweep_label]))
            ax.plot(
                [float(r[sweep_label]) for r in d],
                [float(r["std_test_accuracy"]) for r in d],
                marker=".",
                label=f"b={b}",
            )
        ax.set_title(f"{model_type} / {act} -- std test accuracy")
        ax.set_xlabel(sweep_label)
        ax.set_ylabel("std test accuracy")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        # 3) mean test loss
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for b in batch_sizes:
            d = sorted([r for r in act_rows if int(r["batch_size"]) == b], key=lambda r: float(r[sweep_label]))
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_test_loss"]) for r in d],
                yerr=[float(r["stderr_test_loss"]) for r in d],
                marker=".",
                label=f"b={b}",
            )
        ax.set_title(f"{model_type} / {act} -- mean test loss")
        ax.set_xlabel(sweep_label)
        ax.set_ylabel("mean test loss")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

        # 4) mean train loss
        fig, ax = plt.subplots(1, 1, figsize=(5, 3.5))
        for b in batch_sizes:
            d = sorted([r for r in act_rows if int(r["batch_size"]) == b], key=lambda r: float(r[sweep_label]))
            ax.errorbar(
                [float(r[sweep_label]) for r in d],
                [float(r["mean_train_loss"]) for r in d],
                yerr=[float(r["stderr_train_loss"]) for r in d],
                marker=".",
                label=f"b={b}",
            )
        ax.set_title(f"{model_type} / {act} -- mean train loss")
        ax.set_xlabel(sweep_label)
        ax.set_ylabel("mean train loss")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()


